In [1]:
# pip install rpy2

In [2]:
# import rpy2
# import rpy2.robjects as robjects

# ## To aid in printing HTML in notebooks
# import rpy2.ipython.html
# rpy2.ipython.html.init_printing()

# ## To see plots in an output cell
# # from rpy2.ipython.ggplot import image_png

In [3]:
# %load_ext rpy2.ipython

In [4]:
# with open("synthetic_exp.R", "r") as f:
#     r_code = f.read()

# # Install the riskRegression package if not already installed
# # robjects.r('if (!requireNamespace("riskRegression", quietly = TRUE)) install.packages("riskRegression", repos = "http://cran.us.r-project.org")')
# # Add this in a new cell and run it first
# !sudo apt-get update && sudo apt-get install -y r-cran-riskregression


In [5]:
# !sudo apt-get update && sudo apt-get install -y r-cran-riskregression r-cran-comparec

In [6]:
# robjects.r(r_code)

In [7]:
!pip install scikit-survival

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.6 MB/s eta 0:00:00
  Created wheel for ecos: filename=ecos-2.0.14-cp313-cp313-linux_x86_64.whl size=203999 sha256=244b18873e8cbbf0ca95a26a6e7ce2486b95c5849fb4d0438beb3fc090cfbfe5
  Stored in directory: /root/.cache/pip/wheels/6b/82/0b/4bb5aa6c4618f367601a45db8710205d56858808423974e92a
Successfully built ecos
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# from __future__ import annotations

# import os
# import shutil
# import subprocess
# from dataclasses import dataclass
# from typing import Dict, List, Tuple

# import numpy as np
# import pandas as pd
# import tensorflow as tf
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.decomposition import PCA
# from sksurv.metrics import concordance_index_censored, integrated_brier_score
# from sksurv.linear_model.coxph import BreslowEstimator


# ENCODING_DIM: int = 2
# EPOCHS: int = 400
# LR: float = 1e-2
# R_ENCODER_SCRIPT = "/content/drive/MyDrive/Colab Notebooks/rcode.R"


# def category_labels(K: int) -> List[str]:
#     return [f"C_{{1,{i}}}" for i in range(1, K + 1)]


# def make_probs_long_tail(K: int, head_prop: float = 0.6, head_frac: float = 0.4) -> np.ndarray:
#     n_head = max(2, int(np.floor(K * head_frac)))
#     n_tail = K - n_head
#     p_head = np.repeat(head_prop / n_head, n_head)

#     if n_tail > 0:
#         tail_raw = 1 / (np.arange(1, n_tail + 1) ** 1.2)
#         p_tail = (1 - head_prop) * tail_raw / tail_raw.sum()
#         p = np.concatenate([p_head, p_tail])
#     else:
#         p = p_head

#     return p / p.sum()


# def make_clustered_beta(K: int, n_clusters: int = 4, beta_sd: float = 0.6) -> Tuple[np.ndarray, np.ndarray]:
#     cluster_risks = [-2,-1,1,2]
#     assignments = np.array_split(np.arange(K), n_clusters)
#     beta_g = np.zeros(K)
#     cluster_labels = np.zeros(K, dtype=int)

#     for cid, idx in enumerate(assignments):
#         beta_g[idx] = cluster_risks[cid] + np.random.normal(0, 0.05, len(idx))
#         cluster_labels[idx] = cid

#     return beta_g, cluster_labels


# def generate_survival_dataset(
#     n: int,
#     K: int,
#     beta_g: np.ndarray,
#     lam: float = 0.08,
#     censor_prop: float = 0.4,
# ) -> pd.DataFrame:
#     probs = make_probs_long_tail(K, head_prop=0.6, head_frac=0.4)
#     g = np.random.choice(np.arange(K), size=n, replace=True, p=probs)

#     cont_1 = np.random.normal(size=n)
#     cont_2 = np.random.normal(size=n)
#     lp = beta_g[g] + 0.5 * cont_1 - 0.4 * cont_2

#     T_event = np.random.exponential(scale=1 / (lam * np.exp(lp)))

#     if censor_prop <= 0:
#         T_cens = np.full(n, np.inf)
#     else:
#         lambda_c = (censor_prop / (1 - censor_prop)) * lam
#         T_cens = np.random.exponential(scale=1 / lambda_c, size=n)

#     labels = np.array(category_labels(K))

#     return pd.DataFrame({
#         "cont_1": cont_1,
#         "cont_2": cont_2,
#         "cat_1": labels[g],
#         "Time": np.minimum(T_event, T_cens),
#         "Event": (T_event <= T_cens).astype(int),
#     })


# def run_r_km_greedy_encoder(
#     train_df: pd.DataFrame,
#     test_df: pd.DataFrame,
#     work_dir: str,
# ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
#     os.makedirs(work_dir, exist_ok=True)

#     train_csv = os.path.join(work_dir, "train.csv")
#     test_csv = os.path.join(work_dir, "test.csv")
#     enc_dir = os.path.join(work_dir, "encoded")

#     if os.path.exists(enc_dir):
#         shutil.rmtree(enc_dir)
#     os.makedirs(enc_dir, exist_ok=True)

#     train_df.to_csv(train_csv, index=False)
#     test_df.to_csv(test_csv, index=False)

#     subprocess.run(
#         [
#             "Rscript",
#             R_ENCODER_SCRIPT,
#             train_csv,
#             test_csv,
#             enc_dir,
#         ],
#         check=True,
#     )

#     X_train = pd.read_csv(
#         os.path.join(enc_dir, "X_train_km_greedy.csv")
#     ).values.astype("float32")

#     X_test = pd.read_csv(
#         os.path.join(enc_dir, "X_test_km_greedy.csv")
#     ).values.astype("float32")

#     # Per-category KM profiles: one row per training category
#     cat_profiles_df = pd.read_csv(
#         os.path.join(enc_dir, "category_km_profiles.csv"),
#         index_col=0,
#     )
#     X_cat_profiles = cat_profiles_df.values.astype("float32")
#     cat_profile_labels = list(cat_profiles_df.index)

#     return X_train, X_test, X_cat_profiles, cat_profile_labels


# def negative_cox_partial_log_likelihood(
#     time: tf.Tensor,
#     event: tf.Tensor,
#     risk_score: tf.Tensor,
# ) -> tf.Tensor:
#     time = tf.reshape(tf.cast(time, tf.float32), [-1])
#     event = tf.reshape(tf.cast(event, tf.float32), [-1])
#     risk = tf.reshape(tf.cast(risk_score, tf.float32), [-1])

#     order = tf.argsort(time, direction="DESCENDING")
#     event_s = tf.gather(event, order)
#     risk_s = tf.gather(risk, order)

#     log_cumrisk = tf.math.log(tf.cumsum(tf.exp(risk_s)) + 1e-8)

#     neg_pll = -tf.reduce_sum((risk_s - log_cumrisk) * event_s) / (
#         tf.reduce_sum(event_s) + 1e-8
#     )

#     return neg_pll


# class SmallEncoder(tf.keras.layers.Layer):
#     def __init__(self, hidden_units: List[int], last_activation: str = "sigmoid", **kwargs):
#         super().__init__(**kwargs)
#         activations = ["sigmoid"] * (len(hidden_units) - 1) + [last_activation]
#         self._dense_layers = [
#             tf.keras.layers.Dense(u, activation=a)
#             for u, a in zip(hidden_units, activations)
#         ]

#     def call(self, x, training=False):
#         h = x
#         for layer in self._dense_layers:
#             h = layer(h)
#         return h


# class NeuralCox(tf.keras.Model):
#     def __init__(self, encoder, cat_input_dim: int, cont_dim: int, **kwargs):
#         super().__init__(**kwargs)
#         self.encoder = encoder
#         self.cat_input_dim = cat_input_dim
#         self.cont_dim = cont_dim
#         self.risk_layer = tf.keras.layers.Dense(1, activation=None)

#     def call(self, inputs, training=False):
#         cat_x, cont_x = inputs
#         z = self.encoder(cat_x, training=training) if self.encoder is not None else cat_x
#         return self.risk_layer(tf.concat([z, cont_x], axis=1))

#     def encode(self, X_cat: np.ndarray) -> np.ndarray:
#         if self.encoder is None:
#             return X_cat

#         return self.encoder(
#             tf.constant(X_cat, dtype=tf.float32),
#             training=False,
#         ).numpy()


# def train_neural_cox(
#     model: NeuralCox,
#     X_cat_train: np.ndarray,
#     X_cont_train: np.ndarray,
#     time_train: np.ndarray,
#     event_train: np.ndarray,
#     epochs: int = EPOCHS,
#     lr: float = LR,
#     verbose: int = 1,
# ) -> NeuralCox:
#     optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

#     cat_t = tf.constant(X_cat_train, dtype=tf.float32)
#     cont_t = tf.constant(X_cont_train, dtype=tf.float32)
#     time_t = tf.constant(time_train, dtype=tf.float32)
#     event_t = tf.constant(event_train, dtype=tf.float32)

#     for epoch in range(epochs):
#         with tf.GradientTape() as tape:
#             risk = model([cat_t, cont_t], training=True)
#             loss = negative_cox_partial_log_likelihood(time_t, event_t, risk)

#         grads = tape.gradient(loss, model.trainable_variables)
#         optimizer.apply_gradients(zip(grads, model.trainable_variables))

#         if verbose and (epoch + 1) % 50 == 0:
#             print(f"  epoch={epoch + 1:04d}  loss={float(loss):.4f}")

#     return model


# def predict_risk(
#     model: NeuralCox,
#     X_cat: np.ndarray,
#     X_cont: np.ndarray,
# ) -> np.ndarray:
#     return model(
#         [
#             tf.constant(X_cat, dtype=tf.float32),
#             tf.constant(X_cont, dtype=tf.float32),
#         ],
#         training=False,
#     ).numpy().reshape(-1)




# def make_survival_array(event: np.ndarray, time: np.ndarray) -> np.ndarray:
#     """Create the structured survival array expected by scikit-survival."""
#     return np.array(
#         list(zip(event.astype(bool), time.astype(float))),
#         dtype=[("event", bool), ("time", float)],
#     )


# def make_ibs_time_grid(
#     time_train: np.ndarray,
#     time_test: np.ndarray,
#     n_times: int = 100,
# ) -> np.ndarray:
#     """Choose an evaluation grid that stays inside the train/test follow-up range.

#     integrated_brier_score uses IPCW estimated from the training data, so the
#     evaluation times must not extend beyond the observed training follow-up.
#     We also avoid the exact endpoints because scikit-survival expects times to
#     lie inside the observed test-time range.
#     """
#     time_train = np.asarray(time_train, dtype=float)
#     time_test = np.asarray(time_test, dtype=float)

#     lower = max(np.min(time_test), 1e-8)
#     upper = min(np.max(time_test), np.max(time_train))

#     eps = 1e-8 * max(1.0, upper)
#     lower = lower + eps
#     upper = upper - eps

#     if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
#         return np.array([], dtype=float)

#     return np.linspace(lower, upper, n_times)


# def estimate_survival_probabilities_from_cox_risk(
#     train_risk: np.ndarray,
#     test_risk: np.ndarray,
#     event_train: np.ndarray,
#     time_train: np.ndarray,
#     times: np.ndarray,
# ) -> np.ndarray:
#     """Estimate S(t | x) from Cox risk scores using Breslow baseline survival."""
#     breslow = BreslowEstimator()
#     breslow.fit(
#         linear_predictor=np.asarray(train_risk, dtype=float),
#         event=np.asarray(event_train, dtype=bool),
#         time=np.asarray(time_train, dtype=float),
#     )

#     surv_fns = breslow.get_survival_function(np.asarray(test_risk, dtype=float))
#     surv_probs = np.asarray([fn(times) for fn in surv_fns], dtype=float)

#     # Numerical safety for Brier-score calculation.
#     return np.clip(surv_probs, 1e-8, 1.0)


# def compute_integrated_brier_score_for_model(
#     model: NeuralCox,
#     X_cat_tr: np.ndarray,
#     X_cont_tr: np.ndarray,
#     X_cat_te: np.ndarray,
#     X_cont_te: np.ndarray,
#     time_train: np.ndarray,
#     event_train: np.ndarray,
#     time_test: np.ndarray,
#     event_test: np.ndarray,
#     n_times: int = 100,
# ) -> float:
#     """Compute IBS from the model's Cox risk scores."""
#     times = make_ibs_time_grid(time_train, time_test, n_times=n_times)

#     if len(times) == 0:
#         return float("nan")

#     y_train = make_survival_array(event_train, time_train)
#     y_test = make_survival_array(event_test, time_test)

#     train_risk = predict_risk(model, X_cat_tr, X_cont_tr)
#     test_risk = predict_risk(model, X_cat_te, X_cont_te)

#     surv_probs = estimate_survival_probabilities_from_cox_risk(
#         train_risk=train_risk,
#         test_risk=test_risk,
#         event_train=event_train,
#         time_train=time_train,
#         times=times,
#     )

#     return float(integrated_brier_score(y_train, y_test, surv_probs, times))


# def score_model_sksurv(
#     model: NeuralCox,
#     X_cat_tr: np.ndarray,
#     X_cont_tr: np.ndarray,
#     X_cat_te: np.ndarray,
#     X_cont_te: np.ndarray,
#     time_train: np.ndarray,
#     event_train: np.ndarray,
#     time_test: np.ndarray,
#     event_test: np.ndarray,
# ) -> dict:
#     risk_te = predict_risk(model, X_cat_te, X_cont_te)

#     ci_result = concordance_index_censored(
#         event_test.astype(bool),
#         time_test,
#         risk_te,
#     )

#     try:
#         ibs = compute_integrated_brier_score_for_model(
#             model=model,
#             X_cat_tr=X_cat_tr,
#             X_cont_tr=X_cont_tr,
#             X_cat_te=X_cat_te,
#             X_cont_te=X_cont_te,
#             time_train=time_train,
#             event_train=event_train,
#             time_test=time_test,
#             event_test=event_test,
#         )
#     except Exception as exc:
#         print(f"  Warning: IBS could not be computed for {model.name}: {exc}")
#         ibs = float("nan")

#     return {
#         "cindex": float(ci_result[0]),
#         "ibs": ibs,
#     }


# def build_ohe_arrays(
#     train_df: pd.DataFrame,
#     test_df: pd.DataFrame,
#     cat_col: str = "cat_1",
# ) -> Tuple[np.ndarray, np.ndarray, OneHotEncoder]:
#     enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
#     X_train = enc.fit_transform(train_df[[cat_col]]).astype("float32")
#     X_test = enc.transform(test_df[[cat_col]]).astype("float32")
#     return X_train, X_test, enc


# def build_cont_arrays(
#     train_df: pd.DataFrame,
#     test_df: pd.DataFrame,
#     cont_cols: List[str] = ("cont_1", "cont_2"),
# ) -> Tuple[np.ndarray, np.ndarray]:
#     scaler = StandardScaler()
#     X_train = scaler.fit_transform(train_df[list(cont_cols)]).astype("float32")
#     X_test = scaler.transform(test_df[list(cont_cols)]).astype("float32")
#     return X_train, X_test


# def _enc_input_for_model(
#     name: str,
#     K: int,
#     km_profiles_input: np.ndarray,
#     ohe_encoder: OneHotEncoder,
# ) -> np.ndarray:
#     if "km" in name:
#         return km_profiles_input

#     # Use the actual fitted encoder to guarantee matching dimensions
#     labels = category_labels(K)
#     ohe_matrix = ohe_encoder.transform(pd.DataFrame({"cat_1": labels})).astype("float32")
#     return ohe_matrix


# def plot_encoding_diagnostics(
#     K: int,
#     cluster_labels: np.ndarray,
#     trained_models: Dict[str, NeuralCox],
#     km_profiles_input: np.ndarray,
#     enc_dim: int,
#     out_dir: str,
#     ohe_encoder: OneHotEncoder,
#     train_df: pd.DataFrame,
#     km_cat_labels: List[str],
# ) -> None:
#     import matplotlib.pyplot as plt
#     import matplotlib as mpl

#     n_clusters = len(np.unique(cluster_labels))

#     try:
#         cmap = mpl.colormaps["tab10"]
#     except AttributeError:
#         cmap = plt.cm.get_cmap("tab10")

#     cluster_colors = [cmap(c / 10.0) for c in range(n_clusters)]

#     plot_order = [
#         ("km_greedy", (0, slice(0, 2))),
#         ("joe_ohe_sigmoid", (1, 0)),
#         ("joe_ohe_linear", (1, 1)),
#         ("joe_km_greedy_sigmoid", (2, 0)),
#         ("joe_km_greedy_linear", (2, 1)),
#     ]

#     plot_items = [
#         (name, pos, trained_models[name])
#         for name, pos in plot_order
#         if name in trained_models
#     ]

#     if not plot_items:
#         print("  No plottable encoder models found; skipping encoding plot.")
#         return

#     fig = plt.figure(figsize=(9, 10))
#     gs = fig.add_gridspec(3, 2)

#     axes = []
#     for name, pos, model in plot_items:
#         ax = fig.add_subplot(gs[pos])
#         axes.append((ax, name, model))

#     rng = np.random.default_rng(42)

#     for ax, name, model in axes:
#         enc_input = _enc_input_for_model(name, K, km_profiles_input, ohe_encoder)
#         encodings = model.encode(enc_input)

#         # Per-category train counts
#         counts = (
#             train_df.groupby("cat_1")
#             .agg(
#                 n_events=("Event", "sum"),
#                 n_censored=("Event", lambda x: (x == 0).sum()),
#             )
#             .reset_index()
#             .rename(columns={"cat_1": "category"})
#         )

#         cat_names = km_cat_labels if "km" in name else category_labels(K)

#         # Save encodings to CSV for inspection
#         encodings_df = pd.DataFrame(
#             encodings,
#             columns=[f"dim{i+1}" for i in range(encodings.shape[1])],
#         )
#         encodings_df["category"] = cat_names
#         encodings_df["cluster"] = cluster_labels
#         encodings_df = encodings_df.merge(counts, on="category", how="left")
#         encodings_df.to_csv(os.path.join(out_dir, f"encodings_{name}.csv"), index=False)

#         if enc_dim == 1:
#             x = encodings.reshape(-1)
#             y = rng.uniform(-0.3, 0.3, size=K)

#             for cid in range(n_clusters):
#                 mask = cluster_labels == cid
#                 ax.scatter(
#                     x[mask],
#                     y[mask],
#                     color=cluster_colors[cid],
#                     label=f"Cluster {cid}",
#                     s=55,
#                     alpha=0.85,
#                     edgecolors="white",
#                     linewidths=0.4,
#                 )

#             ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
#             ax.set_xlabel("Learned encoding", fontsize=11)
#             ax.set_yticks([])
#             ax.set_ylabel("")

#         elif enc_dim == 2:
#             for cid in range(n_clusters):
#                 mask = cluster_labels == cid
#                 ax.scatter(
#                     encodings[mask, 0],
#                     encodings[mask, 1],
#                     color=cluster_colors[cid],
#                     label=f"Cluster {cid}",
#                     s=55,
#                     alpha=0.85,
#                     edgecolors="white",
#                     linewidths=0.4,
#                 )

#             ax.set_xlabel("Encoding dim 1", fontsize=11)
#             ax.set_ylabel("Encoding dim 2", fontsize=11)

#         else:
#             pca_enc = PCA(n_components=2)
#             enc_2d = pca_enc.fit_transform(encodings)
#             vr = pca_enc.explained_variance_ratio_ * 100

#             for cid in range(n_clusters):
#                 mask = cluster_labels == cid
#                 ax.scatter(
#                     enc_2d[mask, 0],
#                     enc_2d[mask, 1],
#                     color=cluster_colors[cid],
#                     label=f"Cluster {cid}",
#                     s=55,
#                     alpha=0.85,
#                     edgecolors="white",
#                     linewidths=0.4,
#                 )

#             ax.set_xlabel(f"Encoding PC1 ({vr[0]:.1f}% var)", fontsize=11)
#             ax.set_ylabel(f"Encoding PC2 ({vr[1]:.1f}% var)", fontsize=11)

#         ax.set_title(name.replace("_", " "), fontsize=11)
#         ax.legend(fontsize=8, framealpha=0.6)

#     dim_label = "1D" if enc_dim == 1 else "2D" if enc_dim == 2 else f"{enc_dim}D shown by PCA"

#     # fig.suptitle(
#     #     f"Category encodings/profiles per method ({dim_label}), coloured by true cluster",
#     #     fontsize=12,
#     #     y=1.02,
#     # )

#     fig.tight_layout()
#     fig.savefig(os.path.join(out_dir, "encoding_clusters.png"), dpi=150, bbox_inches="tight")
#     plt.close(fig)

#     print("  Saved encoding_clusters.png")


# dataclass
# class ExperimentConfig:
#     n_train: int = 500
#     n_test: int = 5000
#     K: int = 40
#     censor_prop: float = 0.4
#     encoding_dim: int = ENCODING_DIM
#     epochs: int = EPOCHS
#     lr: float = LR


# def run_one_experiment(
#     cfg: ExperimentConfig,
#     beta_g: np.ndarray,
#     cluster_labels: np.ndarray,
#     seed: int,
#     return_models: bool = False,
#     out_dir: str = "results_ch5_comparison",
# ) -> Dict:
#     train_df = generate_survival_dataset(
#         cfg.n_train,
#         cfg.K,
#         beta_g,
#         censor_prop=cfg.censor_prop,
#     )

#     test_df = generate_survival_dataset(
#         cfg.n_test,
#         cfg.K,
#         beta_g,
#         censor_prop=cfg.censor_prop,
#     )

#     time_train = train_df["Time"].to_numpy()
#     event_train = train_df["Event"].to_numpy()
#     time_test = test_df["Time"].to_numpy()
#     event_test = test_df["Event"].to_numpy()

#     X_cont_train, X_cont_test = build_cont_arrays(train_df, test_df)
#     X_ohe_train, X_ohe_test, ohe_encoder = build_ohe_arrays(train_df, test_df)

#     r_work_dir = os.path.join(
#         out_dir,
#         "r_km_greedy_cache",
#         f"K_{cfg.K}_censor_{cfg.censor_prop}_seed_{seed}",
#     )

#     X_km_train, X_km_test, X_km_cat_profiles, km_cat_labels = run_r_km_greedy_encoder(
#         train_df=train_df,
#         test_df=test_df,
#         work_dir=r_work_dir,
#     )

#     km_dim = X_km_train.shape[1]
#     ohe_dim = X_ohe_train.shape[1]
#     enc_dim = cfg.encoding_dim

#     results: Dict = {
#         "n_train": cfg.n_train,
#         "K": cfg.K,
#         "censor_prop": cfg.censor_prop,
#         "encoding_dim": enc_dim,
#         "events_train": int(train_df["Event"].sum()),
#         "km_greedy_dim": int(km_dim),
#     }

#     trained_models: Dict[str, NeuralCox] = {}

#     print("NN OHE:")
#     model_nn = NeuralCox(
#         encoder=None,
#         cat_input_dim=ohe_dim,
#         cont_dim=2,
#         name="nn_ohe",
#     )

#     train_neural_cox(
#         model_nn,
#         X_ohe_train,
#         X_cont_train,
#         time_train,
#         event_train,
#         epochs=cfg.epochs,
#         lr=cfg.lr,
#     )

#     scores = score_model_sksurv(
#         model_nn,
#         X_ohe_train,
#         X_cont_train,
#         X_ohe_test,
#         X_cont_test,
#         time_train,
#         event_train,
#         time_test,
#         event_test,
#     )

#     results["cindex_nn_ohe"] = scores["cindex"]
#     results["ibs_nn_ohe"] = scores["ibs"]
#     trained_models["nn_ohe"] = model_nn

#     for variant, hidden_units, last_act in [
#         ("sigmoid", [3, enc_dim], "sigmoid"),
#         ("linear", [enc_dim], "linear"),
#     ]:
#         print(f"JOE KM-GREEDY {hidden_units} {variant}")

#         enc = SmallEncoder(
#             hidden_units,
#             last_activation=last_act,
#             name=f"km_greedy_enc_{variant}",
#         )

#         model = NeuralCox(
#             encoder=enc,
#             cat_input_dim=km_dim,
#             cont_dim=2,
#             name=f"joe_km_greedy_{variant}",
#         )

#         train_neural_cox(
#             model,
#             X_km_train,
#             X_cont_train,
#             time_train,
#             event_train,
#             epochs=cfg.epochs,
#             lr=cfg.lr,
#         )

#         key = f"joe_km_greedy_{variant}"

#         scores = score_model_sksurv(
#             model,
#             X_km_train,
#             X_cont_train,
#             X_km_test,
#             X_cont_test,
#             time_train,
#             event_train,
#             time_test,
#             event_test,
#         )

#         results[f"cindex_{key}"] = scores["cindex"]
#         results[f"ibs_{key}"] = scores["ibs"]
#         trained_models[key] = model

#     for variant, hidden_units, last_act in [
#         ("sigmoid", [3, enc_dim], "sigmoid"),
#         ("linear", [enc_dim], "linear"),
#     ]:
#         print(f"JOE OHE {hidden_units} {variant}")

#         enc = SmallEncoder(
#             hidden_units,
#             last_activation=last_act,
#             name=f"ohe_enc_{variant}",
#         )

#         model = NeuralCox(
#             encoder=enc,
#             cat_input_dim=ohe_dim,
#             cont_dim=2,
#             name=f"joe_ohe_{variant}",
#         )

#         train_neural_cox(
#             model,
#             X_ohe_train,
#             X_cont_train,
#             time_train,
#             event_train,
#             epochs=cfg.epochs,
#             lr=cfg.lr,
#         )

#         key = f"joe_ohe_{variant}"

#         scores = score_model_sksurv(
#             model,
#             X_ohe_train,
#             X_cont_train,
#             X_ohe_test,
#             X_cont_test,
#             time_train,
#             event_train,
#             time_test,
#             event_test,
#         )

#         results[f"cindex_{key}"] = scores["cindex"]
#         results[f"ibs_{key}"] = scores["ibs"]
#         trained_models[key] = model

#     base_ci = results["cindex_nn_ohe"]


#     print("KM-GREEDY:")

#     model_km = NeuralCox(
#         encoder=None,
#         cat_input_dim=km_dim,
#         cont_dim=2,
#         name="km_greedy",
#     )

#     train_neural_cox(
#         model_km,
#         X_km_train,
#         X_cont_train,
#         time_train,
#         event_train,
#         epochs=cfg.epochs,
#         lr=cfg.lr,
#     )

#     scores = score_model_sksurv(
#         model_km,
#         X_km_train,
#         X_cont_train,
#         X_km_test,
#         X_cont_test,
#         time_train,
#         event_train,
#         time_test,
#         event_test,
#     )

#     results["cindex_km_greedy"] = scores["cindex"]
#     results["ibs_km_greedy"] = scores["ibs"]
#     trained_models["km_greedy"] = model_km






#     base_ibs = results["ibs_nn_ohe"]

#     for key in [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]:
#         # C-index: higher is better, so gain is method - NN OHE baseline.
#         results[f"gain_cindex_{key}"] = results[f"cindex_{key}"] - base_ci

#         # IBS: lower is better, so define improvement as baseline - method.
#         # Positive values therefore mean better than NN OHE, matching C-index gains.
#         results[f"gain_ibs_{key}"] = base_ibs - results[f"ibs_{key}"]

#     if return_models:
#         results["_trained_models"] = trained_models
#         results["_X_km_cat_profiles"] = X_km_cat_profiles
#         results["_km_cat_labels"] = km_cat_labels
#         results["_ohe_encoder"] = ohe_encoder
#         results["_train_df"] = train_df

#     return results


# def run_grid(K_grid, censor_grid, seeds, out_dir: str) -> pd.DataFrame:
#     rows = []

#     for K in K_grid:
#         for censor in censor_grid:
#             for seed in seeds:
#                 print(f"  K={K}  censor={censor}  seed={seed}")

#                 np.random.seed(seed)
#                 tf.random.set_seed(seed)

#                 beta_g, cluster_labels = make_clustered_beta(K, n_clusters=4)

#                 cfg = ExperimentConfig(K=K, censor_prop=censor)

#                 try:
#                     res = run_one_experiment(
#                         cfg,
#                         beta_g,
#                         cluster_labels,
#                         seed=seed,
#                         return_models=True,
#                         out_dir=out_dir,
#                     )

#                     res["seed"] = seed

#                     trained_models = res.pop("_trained_models")
#                     km_cat_profiles = res.pop("_X_km_cat_profiles")
#                     km_cat_labels = res.pop("_km_cat_labels")
#                     ohe_encoder = res.pop("_ohe_encoder")
#                     train_df_for_plot = res.pop("_train_df")

#                     rows.append(res)

#                     # Build correctly ordered cluster labels for the categories
#                     # that the R encoder actually saw (may be fewer than K if
#                     # some categories never appeared in training)
#                     label_to_cluster = {
#                         category_labels(K)[i]: int(cluster_labels[i]) for i in range(K)
#                     }
#                     km_plot_cluster_labels = np.array([
#                         label_to_cluster.get(lv, -1) for lv in km_cat_labels
#                     ])

#                     vis_dir = os.path.join(
#                         out_dir,
#                         "encoding_visualisations",
#                         f"K_{K}_censor_{censor}_seed_{seed}",
#                     )
#                     os.makedirs(vis_dir, exist_ok=True)

#                     plot_encoding_diagnostics(
#                         K=len(km_cat_labels),
#                         cluster_labels=km_plot_cluster_labels,
#                         trained_models=trained_models,
#                         km_profiles_input=km_cat_profiles,
#                         enc_dim=cfg.encoding_dim,
#                         out_dir=vis_dir,
#                         ohe_encoder=ohe_encoder,
#                         train_df=train_df_for_plot,
#                         km_cat_labels=km_cat_labels,
#                     )

#                 except Exception as exc:
#                     rows.append({
#                         "K": K,
#                         "censor_prop": censor,
#                         "seed": seed,
#                         "km_greedy_dim": None,
#                         "error": str(exc),
#                     })

#     return pd.DataFrame(rows)


# def plot_metric_gain_boxplots(
#     df_res: pd.DataFrame,
#     out_dir: str,
# ) -> None:
#     """Save side-by-side seed-level gain plots for C-index and IBS.

#     C-index gain is defined as method C-index minus NN OHE C-index.
#     IBS gain is defined as NN OHE IBS minus method IBS, because lower IBS is better.
#     Therefore, positive values are better than NN OHE for both plots.
#     """
#     import matplotlib.pyplot as plt

#     method_order = [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]

#     method_labels = {
#         "km_greedy": "KM-GREEDY",
#         "joe_km_greedy_sigmoid": "JOE KM sigmoid",
#         "joe_km_greedy_linear": "JOE KM linear",
#         "joe_ohe_sigmoid": "JOE OHE sigmoid",
#         "joe_ohe_linear": "JOE OHE linear",
#     }

#     plot_specs = [
#         (
#             "gain_cindex",
#             "C-index gain vs NN OHE",
#             "C-index gain\n(method - NN OHE)",
#             "cindex_gain_boxplot.png",
#         ),
#         (
#             "gain_ibs",
#             "IBS improvement vs NN OHE",
#             "IBS improvement\n(NN OHE - method)",
#             "ibs_gain_boxplot.png",
#         ),
#     ]

#     for prefix, title, ylabel, filename in plot_specs:
#         cols = [f"{prefix}_{m}" for m in method_order if f"{prefix}_{m}" in df_res.columns]
#         labels = [method_labels[m] for m in method_order if f"{prefix}_{m}" in df_res.columns]

#         if not cols:
#             print(f"  No columns found for {prefix}; skipping gain plot.")
#             continue

#         data = [df_res[c].dropna().to_numpy() for c in cols]
#         keep = [i for i, arr in enumerate(data) if len(arr) > 0]

#         if not keep:
#             print(f"  No non-missing data found for {prefix}; skipping gain plot.")
#             continue

#         data = [data[i] for i in keep]
#         labels = [labels[i] for i in keep]

#         fig, ax = plt.subplots(figsize=(1.6 * len(data) + 3, 4.8))
#         bp = ax.boxplot(
#             data,
#             labels=labels,
#             showmeans=True,
#             patch_artist=True,
#         )

#         for box in bp["boxes"]:
#             box.set_alpha(0.65)

#         ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
#         ax.set_title(title, fontsize=12)
#         ax.set_ylabel(ylabel, fontsize=11)
#         ax.tick_params(axis="x", labelrotation=30)
#         ax.grid(axis="y", alpha=0.25)

#         fig.tight_layout()
#         fig.savefig(os.path.join(out_dir, filename), dpi=150, bbox_inches="tight")
#         plt.close(fig)

#         print(f"  Saved {filename}")


# if __name__ == "__main__":
#     out_dir = "results_ch5_comparison"
#     os.makedirs(out_dir, exist_ok=True)

#     K_grid = [40]
#     censor_grid = [0.3]
#     seeds = range(50)

#     df_res = run_grid(K_grid, censor_grid, seeds, out_dir)

#     df_res.to_csv(
#         os.path.join(out_dir, "chapter5_comparison_results.csv"),
#         index=False,
#     )

#     plot_metric_gain_boxplots(df_res=df_res, out_dir=out_dir)

#     method_cols = [
#         "cindex_nn_ohe",
#         "cindex_km_greedy",
#         "cindex_joe_km_greedy_sigmoid",
#         "cindex_joe_km_greedy_linear",
#         "cindex_joe_ohe_sigmoid",
#         "cindex_joe_ohe_linear",
#     ]

#     gain_cindex_cols = [
#         "gain_cindex_km_greedy",
#         "gain_cindex_joe_km_greedy_sigmoid",
#         "gain_cindex_joe_km_greedy_linear",
#         "gain_cindex_joe_ohe_sigmoid",
#         "gain_cindex_joe_ohe_linear",
#     ]

#     gain_ibs_cols = [
#         "gain_ibs_km_greedy",
#         "gain_ibs_joe_km_greedy_sigmoid",
#         "gain_ibs_joe_km_greedy_linear",
#         "gain_ibs_joe_ohe_sigmoid",
#         "gain_ibs_joe_ohe_linear",
#     ]

#     gain_cols = gain_cindex_cols + gain_ibs_cols

#     ibs_cols = [
#         "ibs_nn_ohe",
#         "ibs_km_greedy",
#         "ibs_joe_km_greedy_sigmoid",
#         "ibs_joe_km_greedy_linear",
#         "ibs_joe_ohe_sigmoid",
#         "ibs_joe_ohe_linear",
#     ]

#     ok_cols = [c for c in method_cols if c in df_res.columns]

#     summary = (
#         df_res.dropna(subset=ok_cols)
#         .groupby(["K", "censor_prop"], as_index=False)
#         .agg(
#             **{f"mean_{c}": (c, "mean") for c in method_cols if c in df_res.columns},
#             **{f"mean_{g}": (g, "mean") for g in gain_cols if g in df_res.columns},
#             **{f"mean_{i}": (i, "mean") for i in ibs_cols if i in df_res.columns},
#             **({} if "km_greedy_dim" not in df_res.columns else
#                {"mean_km_greedy_dim": ("km_greedy_dim", "mean")}),
#         )
#     )

#     summary.to_csv(
#         os.path.join(out_dir, "chapter5_comparison_summary.csv"),
#         index=False,
#     )

#     print("\n=== Mean C-index by method ===")
#     print(summary.to_string(index=False))

#     print(f"\nAll outputs saved to '{out_dir}/'")

In [10]:
from __future__ import annotations

import os
import shutil
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from sksurv.linear_model.coxph import BreslowEstimator

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    pairwise_distances,
)
from scipy.stats import spearmanr


ENCODING_DIM: int = 2
EPOCHS: int = 400
LR: float = 0.1
R_ENCODER_SCRIPT = "/content/drive/MyDrive/Colab Notebooks/rcode_full_km_multi_cat_ready.R"


def category_labels(K: int) -> List[str]:
    return [f"C_{{1,{i}}}" for i in range(1, K + 1)]


def make_probs_long_tail(K: int, head_prop: float = 0.6, head_frac: float = 0.4) -> np.ndarray:
    n_head = max(2, int(np.floor(K * head_frac)))
    n_tail = K - n_head
    p_head = np.repeat(head_prop / n_head, n_head)

    if n_tail > 0:
        tail_raw = 1 / (np.arange(1, n_tail + 1) ** 1.2)
        p_tail = (1 - head_prop) * tail_raw / tail_raw.sum()
        p = np.concatenate([p_head, p_tail])
    else:
        p = p_head

    return p / p.sum()


def make_clustered_beta(K: int, n_clusters: int = 4, beta_sd: float = 0.6) -> Tuple[np.ndarray, np.ndarray]:
    cluster_risks = [-2,-1,1,2]
    assignments = np.array_split(np.arange(K), n_clusters)
    beta_g = np.zeros(K)
    cluster_labels = np.zeros(K, dtype=int)

    for cid, idx in enumerate(assignments):
        beta_g[idx] = cluster_risks[cid] + np.random.normal(0, 0.05, len(idx))
        cluster_labels[idx] = cid

    return beta_g, cluster_labels


def make_beta_g(K: int, beta_sd: float = 0.4) -> np.ndarray:
    """Create smoothly varying, non-clustered category effects."""
    i = np.arange(1, K + 1)

    beta_g = (
        0.7 * np.sin(2 * np.pi * i / K)
        + 0.35 * np.cos(4 * np.pi * i / K)
        + np.random.normal(0, 0.15, K)
    )

    beta_g = beta_g - beta_g.mean()
    beta_g = beta_g / beta_g.std(ddof=1) * beta_sd

    return beta_g


def make_experiment_effects(
    K: int,
    experiment_type: str,
) -> Tuple[np.ndarray, np.ndarray | None]:
    """Return category effects and optional true cluster labels."""
    if experiment_type == "cluster":
        return make_clustered_beta(K, n_clusters=4)

    if experiment_type == "noncluster":
        return make_beta_g(K, beta_sd=0.4), None

    raise ValueError(
        f"Unknown experiment_type={experiment_type!r}. "
        "Use 'cluster' or 'noncluster'."
    )

# ============================================================
# 17B. ADMINISTRATIVE CENSORING FOR IBS
# ============================================================



def generate_survival_dataset(
    n: int,
    K: int,
    beta_g: np.ndarray,
    lam: float = 0.08,
    censor_prop: float = 0.4,
) -> pd.DataFrame:
    probs = make_probs_long_tail(K, head_prop=0.6, head_frac=0.4)
    g = np.random.choice(np.arange(K), size=n, replace=True, p=probs)

    cont_1 = np.random.normal(size=n)
    cont_2 = np.random.normal(size=n)
    lp = beta_g[g] + 0.5 * cont_1 - 0.4 * cont_2

    T_event = np.random.exponential(scale=1 / (lam * np.exp(lp)))

    if censor_prop <= 0:
        T_cens = np.full(n, np.inf)
    else:
        lambda_c = (censor_prop / (1 - censor_prop)) * lam
        T_cens = np.random.exponential(scale=1 / lambda_c, size=n)

    labels = np.array(category_labels(K))

    return pd.DataFrame({
        "cont_1": cont_1,
        "cont_2": cont_2,
        "cat_1": labels[g],
        "Time": np.minimum(T_event, T_cens),
        "Event": (T_event <= T_cens).astype(int),
    })


def run_r_km_encoder(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    work_dir: str,
) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray,
    np.ndarray, np.ndarray, np.ndarray,
    List[str],
]:
    """Run R and load greedy KM plus full-grid KM representations."""
    os.makedirs(work_dir, exist_ok=True)

    train_csv = os.path.join(work_dir, "train.csv")
    test_csv = os.path.join(work_dir, "test.csv")
    enc_dir = os.path.join(work_dir, "encoded")

    if os.path.exists(enc_dir):
        shutil.rmtree(enc_dir)
    os.makedirs(enc_dir, exist_ok=True)

    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)

    subprocess.run(
        ["Rscript", R_ENCODER_SCRIPT, train_csv, test_csv, enc_dir],
        check=True,
    )

    X_greedy_train = pd.read_csv(
        os.path.join(enc_dir, "X_train_km_greedy.csv")
    ).values.astype("float32")
    X_greedy_test = pd.read_csv(
        os.path.join(enc_dir, "X_test_km_greedy.csv")
    ).values.astype("float32")
    greedy_profiles_df = pd.read_csv(
        os.path.join(enc_dir, "category_km_profiles_greedy.csv"),
        index_col=0,
    )

    X_full_train = pd.read_csv(
        os.path.join(enc_dir, "X_train_km_full.csv")
    ).values.astype("float32")
    X_full_test = pd.read_csv(
        os.path.join(enc_dir, "X_test_km_full.csv")
    ).values.astype("float32")
    full_profiles_df = pd.read_csv(
        os.path.join(enc_dir, "category_km_profiles_full.csv"),
        index_col=0,
    )

    greedy_profiles = greedy_profiles_df.values.astype("float32")
    full_profiles = full_profiles_df.values.astype("float32")
    greedy_labels = list(greedy_profiles_df.index)
    full_labels = list(full_profiles_df.index)

    if greedy_labels != full_labels:
        raise ValueError("Greedy and full KM category order differs.")
    if X_greedy_train.shape[1] != X_greedy_test.shape[1]:
        raise ValueError("Greedy KM train/test dimensions differ.")
    if X_full_train.shape[1] != X_full_test.shape[1]:
        raise ValueError("Full KM train/test dimensions differ.")
    if greedy_profiles.shape[1] != X_greedy_train.shape[1]:
        raise ValueError("Greedy category-profile dimension is inconsistent.")
    if full_profiles.shape[1] != X_full_train.shape[1]:
        raise ValueError("Full category-profile dimension is inconsistent.")

    return (
        X_greedy_train, X_greedy_test, greedy_profiles,
        X_full_train, X_full_test, full_profiles,
        full_labels,
    )

def negative_cox_partial_log_likelihood(
    time: tf.Tensor,
    event: tf.Tensor,
    risk_score: tf.Tensor,
) -> tf.Tensor:
    time = tf.reshape(tf.cast(time, tf.float32), [-1])
    event = tf.reshape(tf.cast(event, tf.float32), [-1])
    risk = tf.reshape(tf.cast(risk_score, tf.float32), [-1])

    order = tf.argsort(time, direction="DESCENDING")
    event_s = tf.gather(event, order)
    risk_s = tf.gather(risk, order)

    log_cumrisk = tf.math.log(tf.cumsum(tf.exp(risk_s)) + 1e-8)

    neg_pll = -tf.reduce_sum((risk_s - log_cumrisk) * event_s) / (
        tf.reduce_sum(event_s) + 1e-8
    )

    return neg_pll


class SmallEncoder(tf.keras.layers.Layer):
    def __init__(self, hidden_units: List[int], last_activation: str = "sigmoid", **kwargs):
        super().__init__(**kwargs)
        activations = ["sigmoid"] * (len(hidden_units) - 1) + [last_activation]
        self._dense_layers = [
            tf.keras.layers.Dense(u, activation=a)
            for u, a in zip(hidden_units, activations)
        ]

    def call(self, x, training=False):
        h = x
        for layer in self._dense_layers:
            h = layer(h)
        return h


class NeuralCox(tf.keras.Model):
    def __init__(self, encoder, cat_input_dim: int, cont_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.cat_input_dim = cat_input_dim
        self.cont_dim = cont_dim
        self.risk_layer = tf.keras.layers.Dense(1, activation=None)

    def call(self, inputs, training=False):
        cat_x, cont_x = inputs
        z = self.encoder(cat_x, training=training) if self.encoder is not None else cat_x
        return self.risk_layer(tf.concat([z, cont_x], axis=1))

    def encode(self, X_cat: np.ndarray) -> np.ndarray:
        if self.encoder is None:
            return X_cat

        return self.encoder(
            tf.constant(X_cat, dtype=tf.float32),
            training=False,
        ).numpy()

def train_neural_cox(
    model: NeuralCox,
    X_cat_train: np.ndarray,
    X_cont_train: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    epochs: int = EPOCHS,
    lr: float = LR,
    verbose: int = 1,
) -> NeuralCox:

    # ========================================================
    # Optimizer
    # ========================================================

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=lr
    )

    # Important because ReduceLROnPlateau normally expects
    # model.optimizer to exist when used with model.fit().
    # We use a custom GradientTape loop, so attach it manually.
    model.optimizer = optimizer


    # ========================================================
    # ReduceLROnPlateau
    # ========================================================

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=20,
        verbose=1,
        mode="min",
        min_delta=1e-4,
        cooldown=0,
        min_lr=1e-5,
    )

    # Attach callback manually because this is not model.fit()
    reduce_lr.set_model(model)

    reduce_lr.on_train_begin()


    # ========================================================
    # Tensors
    # ========================================================

    cat_t = tf.constant(
        X_cat_train,
        dtype=tf.float32,
    )

    cont_t = tf.constant(
        X_cont_train,
        dtype=tf.float32,
    )

    time_t = tf.constant(
        time_train,
        dtype=tf.float32,
    )

    event_t = tf.constant(
        event_train,
        dtype=tf.float32,
    )


    # ========================================================
    # Training
    # ========================================================

    for epoch in range(epochs):

        reduce_lr.on_epoch_begin(epoch)

        with tf.GradientTape() as tape:

            risk = model(
                [
                    cat_t,
                    cont_t,
                ],
                training=True,
            )

            loss = negative_cox_partial_log_likelihood(
                time_t,
                event_t,
                risk,
            )


        # ====================================================
        # Gradients
        # ====================================================

        grads = tape.gradient(
            loss,
            model.trainable_variables,
        )

        # Safely remove None gradients
        grad_var_pairs = [
            (grad, var)
            for grad, var in zip(
                grads,
                model.trainable_variables,
            )
            if grad is not None
        ]

        optimizer.apply_gradients(
            grad_var_pairs
        )


        # ====================================================
        # Current loss
        # ====================================================

        current_loss = float(
            loss.numpy()
        )


        # ====================================================
        # Reduce LR if training loss has plateaued
        # ====================================================

        reduce_lr.on_epoch_end(
            epoch,
            logs={
                "loss": current_loss
            },
        )


        # Read LR AFTER callback has potentially changed it
        current_lr = float(
            tf.keras.backend.get_value(
                optimizer.learning_rate
            )
        )


        # ====================================================
        # Progress output
        # ====================================================

        if (
            verbose
            and (
                epoch == 0
                or (epoch + 1) % 50 == 0
            )
        ):

            print(
                f"  epoch={epoch + 1:04d}  "
                f"loss={current_loss:.5f}  "
                f"lr={current_lr:.7f}"
            )


    reduce_lr.on_train_end()

    return model


def predict_risk(
    model: NeuralCox,
    X_cat: np.ndarray,
    X_cont: np.ndarray,
) -> np.ndarray:
    return model(
        [
            tf.constant(X_cat, dtype=tf.float32),
            tf.constant(X_cont, dtype=tf.float32),
        ],
        training=False,
    ).numpy().reshape(-1)




def make_survival_array(event: np.ndarray, time: np.ndarray) -> np.ndarray:
    """Create the structured survival array expected by scikit-survival."""
    return np.array(
        list(zip(event.astype(bool), time.astype(float))),
        dtype=[("event", bool), ("time", float)],
    )


def make_ibs_time_grid(
    time_train: np.ndarray,
    time_test: np.ndarray,
    n_times: int = 100,
) -> np.ndarray:
    """Choose an evaluation grid that stays inside the train/test follow-up range.

    integrated_brier_score uses IPCW estimated from the training data, so the
    evaluation times must not extend beyond the observed training follow-up.
    We also avoid the exact endpoints because scikit-survival expects times to
    lie inside the observed test-time range.
    """
    time_train = np.asarray(time_train, dtype=float)
    time_test = np.asarray(time_test, dtype=float)

    lower = max(np.min(time_test), 1e-8)
    upper = min(np.max(time_test), np.max(time_train))

    eps = 1e-8 * max(1.0, upper)
    lower = lower + eps
    upper = upper - eps

    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        return np.array([], dtype=float)

    return np.linspace(lower, upper, n_times)


def estimate_survival_probabilities_from_cox_risk(
    train_risk: np.ndarray,
    test_risk: np.ndarray,
    event_train: np.ndarray,
    time_train: np.ndarray,
    times: np.ndarray,
) -> np.ndarray:
    """Estimate S(t | x) from Cox risk scores using Breslow baseline survival."""
    breslow = BreslowEstimator()
    breslow.fit(
        linear_predictor=np.asarray(train_risk, dtype=float),
        event=np.asarray(event_train, dtype=bool),
        time=np.asarray(time_train, dtype=float),
    )

    surv_fns = breslow.get_survival_function(np.asarray(test_risk, dtype=float))
    surv_probs = np.asarray([fn(times) for fn in surv_fns], dtype=float)

    # Numerical safety for Brier-score calculation.
    return np.clip(surv_probs, 1e-8, 1.0)


# def compute_integrated_brier_score_for_model(
#     model: NeuralCox,
#     X_cat_tr: np.ndarray,
#     X_cont_tr: np.ndarray,
#     X_cat_te: np.ndarray,
#     X_cont_te: np.ndarray,
#     time_train: np.ndarray,
#     event_train: np.ndarray,
#     time_test: np.ndarray,
#     event_test: np.ndarray,
#     n_times: int = 100,
# ) -> float:
#     """Compute IBS from the model's Cox risk scores."""

#     train_max = float(np.max(time_train))

#     # Cap any test follow-up beyond the train max -- the IPCW censoring
#     # model (fit on train) is undefined past that point.
#     time_test = np.minimum(time_test, train_max)
#     event_test = np.where(
#         time_test >= train_max,
#         0,
#         event_test,
#     ).astype(int)

#     # Evaluate up to 90% of the shorter follow-up range instead of right
#     # at the boundary. Avoids floating-point ties between the grid and
#     # the observed max, and avoids the noisy last few risk-set points.
#     tau = 0.9 * min(train_max, time_test.max())
#     lower = max(time_test.min(), 1e-6)

#     if tau <= lower:
#         return float("nan")

#     times = np.linspace(lower, tau, n_times)

#     y_train = make_survival_array(event_train, time_train)
#     y_test = make_survival_array(event_test, time_test)

#     train_risk = predict_risk(model, X_cat_tr, X_cont_tr)
#     test_risk = predict_risk(model, X_cat_te, X_cont_te)

#     surv_probs = estimate_survival_probabilities_from_cox_risk(
#         train_risk=train_risk,
#         test_risk=test_risk,
#         event_train=event_train,
#         time_train=time_train,
#         times=times,
#     )

#     return float(integrated_brier_score(y_train, y_test, surv_probs, times))

def compute_integrated_brier_score_for_model(
    model: NeuralCox,
    X_cat_tr: np.ndarray,
    X_cont_tr: np.ndarray,
    X_cat_te: np.ndarray,
    X_cont_te: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    time_test: np.ndarray,
    event_test: np.ndarray,
    n_times: int = 100,
) -> float:
    """
    Mirrors the R evaluation exactly:

        time_interest <- unique(seq(
            quantile(df_test$Time, 0.05, na.rm = TRUE),
            quantile(df_test$Time, 0.95, na.rm = TRUE),
            length.out = 100
        ))

        Score(..., cens.model = "km", cens.data = df_train,
              times = time_interest, summary = "ibs")

    i.e. a 100-point grid spanning the 5th-95th percentile of TEST
    follow-up, with the IPCW censoring distribution fit on TRAIN.
    """

    times = np.unique(
        np.linspace(
            np.quantile(time_test, 0.05),
            np.quantile(time_test, 0.95),
            n_times,
        )
    )

    y_train = make_survival_array(event_train, time_train)
    y_test = make_survival_array(event_test, time_test)

    train_risk = predict_risk(model, X_cat_tr, X_cont_tr)
    test_risk = predict_risk(model, X_cat_te, X_cont_te)

    surv_probs = estimate_survival_probabilities_from_cox_risk(
        train_risk=train_risk,
        test_risk=test_risk,
        event_train=event_train,
        time_train=time_train,
        times=times,
    )

    return float(integrated_brier_score(y_train, y_test, surv_probs, times))


def score_model_sksurv(
    model: NeuralCox,
    X_cat_tr: np.ndarray,
    X_cont_tr: np.ndarray,
    X_cat_te: np.ndarray,
    X_cont_te: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    time_test: np.ndarray,
    event_test: np.ndarray,
) -> dict:
    risk_te = predict_risk(model, X_cat_te, X_cont_te)

    ci_result = concordance_index_censored(
        event_test.astype(bool),
        time_test,
        risk_te,
    )

    try:
        ibs = compute_integrated_brier_score_for_model(
            model=model,
            X_cat_tr=X_cat_tr,
            X_cont_tr=X_cont_tr,
            X_cat_te=X_cat_te,
            X_cont_te=X_cont_te,
            time_train=time_train,
            event_train=event_train,
            time_test=time_test,
            event_test=event_test,
        )
    except Exception as exc:
        print(f"  Warning: IBS could not be computed for {model.name}: {exc}")
        ibs = float("nan")

    return {
        "cindex": float(ci_result[0]),
        "ibs": ibs,
    }


def build_ohe_arrays(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cat_col: str = "cat_1",
) -> Tuple[np.ndarray, np.ndarray, OneHotEncoder]:
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
    X_train = enc.fit_transform(train_df[[cat_col]]).astype("float32")
    X_test = enc.transform(test_df[[cat_col]]).astype("float32")
    return X_train, X_test, enc


def build_cont_arrays(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cont_cols: List[str] = ("cont_1", "cont_2"),
) -> Tuple[np.ndarray, np.ndarray]:
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[list(cont_cols)]).astype("float32")
    X_test = scaler.transform(test_df[list(cont_cols)]).astype("float32")
    return X_train, X_test


def _enc_input_for_model(
    name: str,
    K: int,
    km_greedy_profiles: np.ndarray,
    km_full_profiles: np.ndarray,
    ohe_encoder: OneHotEncoder,
) -> np.ndarray:
    if name == "km_greedy":
        return km_greedy_profiles
    if name.startswith("joe_km_greedy_"):
        # Legacy model names retained, but these neural models now receive
        # the complete fixed-grid KM profile rather than greedy-selected points.
        return km_full_profiles

    labels = category_labels(K)
    return ohe_encoder.transform(
        pd.DataFrame({"cat_1": labels})
    ).astype("float32")


def plot_encoding_diagnostics_cluster(
    K: int,
    cluster_labels: np.ndarray,
    trained_models: Dict[str, NeuralCox],
    km_greedy_profiles: np.ndarray,
    km_full_profiles: np.ndarray,
    enc_dim: int,
    out_dir: str,
    ohe_encoder: OneHotEncoder,
    train_df: pd.DataFrame,
    km_cat_labels: List[str],
) -> None:
    """
    Plot category encodings coloured by their true simulated clusters.

    A single shared cluster legend is placed below the complete figure.
    """
    import os

    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    from matplotlib.lines import Line2D
    from sklearn.decomposition import PCA

    cluster_labels = np.asarray(cluster_labels, dtype=int)
    km_cat_labels = list(km_cat_labels)

    all_category_names = list(category_labels(K))

    if len(cluster_labels) != K:
        raise ValueError(
            f"Expected {K} true cluster labels, "
            f"but received {len(cluster_labels)}."
        )

    # Ground-truth mapping is based on the original simulation order,
    # not the KM-profile ordering.
    cluster_by_category = {
        category: int(cluster)
        for category, cluster in zip(
            all_category_names,
            cluster_labels,
        )
    }

    unique_clusters = np.sort(np.unique(cluster_labels))
    n_clusters = len(unique_clusters)

    try:
        cmap = mpl.colormaps["tab10"]
    except AttributeError:
        cmap = plt.cm.get_cmap("tab10")

    cluster_colors = {
        cluster_id: cmap(i % 10)
        for i, cluster_id in enumerate(unique_clusters)
    }

    # model key, display label, subplot position
    plot_order = [
        ("km_greedy", "KM", (0, slice(0, 2))),
        ("joe_ohe_sigmoid", "JOEOH 3SIG", (1, 0)),
        ("joe_ohe_linear", "JOEOH 1LIN", (1, 1)),
        ("joe_km_greedy_sigmoid", "JOECHAR 3SIG", (2, 0)),
        ("joe_km_greedy_linear", "JOECHAR 1LIN", (2, 1)),
    ]

    plot_items = [
        (
            model_key,
            display_label,
            position,
            trained_models[model_key],
        )
        for model_key, display_label, position in plot_order
        if model_key in trained_models
    ]

    if not plot_items:
        print(
            "  No plottable encoder models found; "
            "skipping encoding plot."
        )
        return

    # Calculate category-level event and censoring counts once.
    counts = (
        train_df.groupby("cat_1")
        .agg(
            n_events=("Event", "sum"),
            n_censored=("Event", lambda x: (x == 0).sum()),
        )
        .reset_index()
        .rename(columns={"cat_1": "category"})
    )

    fig = plt.figure(figsize=(9, 10.5))
    gs = fig.add_gridspec(3, 2)

    axes = []

    for model_key, display_label, position, model in plot_items:
        ax = fig.add_subplot(gs[position])

        axes.append(
            (
                ax,
                model_key,
                display_label,
                model,
            )
        )

    rng = np.random.default_rng(42)

    for ax, model_key, display_label, model in axes:
        enc_input = _enc_input_for_model(
            model_key,
            K,
            km_greedy_profiles,
            km_full_profiles,
            ohe_encoder,
        )

        encodings = np.asarray(
            model.encode(enc_input)
        )

        if encodings.ndim == 1:
            encodings = encodings.reshape(-1, 1)

        # KM-based representations follow km_cat_labels.
        # OHE representations follow category_labels(K).
        if "km" in model_key:
            cat_names = list(km_cat_labels)
        else:
            cat_names = list(category_labels(K))

        if len(cat_names) != encodings.shape[0]:
            raise ValueError(
                f"Category-label mismatch for {model_key}: "
                f"{len(cat_names)} category names but "
                f"{encodings.shape[0]} encoding rows."
            )

        missing_categories = [
            category
            for category in cat_names
            if category not in cluster_by_category
        ]

        if missing_categories:
            raise ValueError(
                f"Could not find true cluster labels for {model_key}. "
                f"Missing categories: {missing_categories}"
            )

        method_cluster_labels = np.asarray(
            [
                cluster_by_category[category]
                for category in cat_names
            ],
            dtype=int,
        )

        # Save category encodings for inspection.
        encodings_df = pd.DataFrame(
            encodings,
            columns=[
                f"dim{i + 1}"
                for i in range(encodings.shape[1])
            ],
        )

        encodings_df["category"] = cat_names
        encodings_df["cluster"] = method_cluster_labels

        encodings_df = encodings_df.merge(
            counts,
            on="category",
            how="left",
        )

        encodings_df.to_csv(
            os.path.join(
                out_dir,
                f"encodings_{model_key}.csv",
            ),
            index=False,
        )

        if enc_dim == 1:
            x = encodings[:, 0]

            y = rng.uniform(
                -0.3,
                0.3,
                size=len(x),
            )

            for cluster_id in unique_clusters:
                mask = method_cluster_labels == cluster_id

                ax.scatter(
                    x[mask],
                    y[mask],
                    color=cluster_colors[cluster_id],
                    s=55,
                    alpha=0.85,
                    edgecolors="white",
                    linewidths=0.4,
                )

            ax.axhline(
                0,
                color="grey",
                linewidth=0.5,
                linestyle="--",
            )

            ax.set_xlabel(
                "Learned encoding",
                fontsize=11,
            )
            ax.set_yticks([])
            ax.set_ylabel("")

        elif enc_dim == 2:
            for cluster_id in unique_clusters:
                mask = method_cluster_labels == cluster_id

                ax.scatter(
                    encodings[mask, 0],
                    encodings[mask, 1],
                    color=cluster_colors[cluster_id],
                    s=55,
                    alpha=0.85,
                    edgecolors="white",
                    linewidths=0.4,
                )

            ax.set_xlabel("Dim 1", fontsize=11)
            ax.set_ylabel("Dim 2", fontsize=11)

        else:
            pca_enc = PCA(n_components=2)
            enc_2d = pca_enc.fit_transform(encodings)

            explained_variance = (
                pca_enc.explained_variance_ratio_ * 100
            )

            for cluster_id in unique_clusters:
                mask = method_cluster_labels == cluster_id

                ax.scatter(
                    enc_2d[mask, 0],
                    enc_2d[mask, 1],
                    color=cluster_colors[cluster_id],
                    s=55,
                    alpha=0.85,
                    edgecolors="white",
                    linewidths=0.4,
                )

            ax.set_xlabel(
                f"Encoding PC1 "
                f"({explained_variance[0]:.1f}% var)",
                fontsize=11,
            )

            ax.set_ylabel(
                f"Encoding PC2 "
                f"({explained_variance[1]:.1f}% var)",
                fontsize=11,
            )

        ax.set_title(
            display_label,
            fontsize=11,
        )

    # One shared legend for the entire figure.
    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markerfacecolor=cluster_colors[cluster_id],
            markeredgecolor="white",
            markeredgewidth=0.4,
            markersize=8,
            label=f"Cluster {cluster_id + 1}",
        )
        for cluster_id in unique_clusters
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=n_clusters,
        frameon=False,
        fontsize=9,
    )

    fig.tight_layout(
        rect=[0, 0.06, 1, 1]
    )

    output_path = os.path.join(
        out_dir,
        "encoding_clusters.pdf",
    )

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(fig)

    print("  Saved encoding_clusters.png")

def plot_encoding_diagnostics_noncluster(
    beta_g: np.ndarray,
    trained_models: Dict[str, NeuralCox],
    km_greedy_profiles: np.ndarray,
    km_full_profiles: np.ndarray,
    enc_dim: int,
    out_dir: str,
    ohe_encoder: OneHotEncoder,
    category_names: List[str],
) -> None:
    """
    Plot learned category encodings coloured by continuous true effects.

    The layout matches the cluster-experiment plot:
        - KM spans the first row
        - JOEOH models occupy the second row
        - JOECHAR models occupy the third row
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    if len(category_names) == 0:
        print(
            "  No categories available; "
            "skipping non-cluster encoding plot."
        )
        return

    beta_g = np.asarray(beta_g, dtype=float)
    category_names = list(category_names)

    if len(beta_g) != len(category_names):
        raise ValueError(
            f"Expected one beta value per category, but received "
            f"{len(beta_g)} beta values and "
            f"{len(category_names)} category names."
        )

    if km_greedy_profiles.shape[0] != len(category_names):
        raise ValueError(
            f"KM profile/category mismatch: "
            f"{km_greedy_profiles.shape[0]} profile rows but "
            f"{len(category_names)} category names."
        )

    norm = plt.Normalize(
        vmin=np.min(beta_g),
        vmax=np.max(beta_g),
    )

    # Model key, displayed title, subplot position.
    # KM spans both columns in the first row.
    plot_order = [
        ("km_greedy", "KM", (0, slice(0, 2))),
        ("joe_ohe_sigmoid", "JOEOH 3SIG", (1, 0)),
        ("joe_ohe_linear", "JOEOH 1LIN", (1, 1)),
        ("joe_km_greedy_sigmoid", "JOECHAR 3SIG", (2, 0)),
        ("joe_km_greedy_linear", "JOECHAR 1LIN", (2, 1)),
    ]

    plot_items = [
        (
            model_key,
            display_label,
            position,
            trained_models[model_key],
        )
        for model_key, display_label, position in plot_order
        if model_key in trained_models
    ]

    if not plot_items:
        print(
            "  No plottable encoder models found; "
            "skipping non-cluster encoding plot."
        )
        return

    fig = plt.figure(
        figsize=(10.5, 10.5),
        constrained_layout=True,
    )

    grid = fig.add_gridspec(3, 2)

    axes = []

    for model_key, display_label, position, model in plot_items:
        ax = fig.add_subplot(grid[position])

        axes.append(
            (
                ax,
                model_key,
                display_label,
                model,
            )
        )

    rng = np.random.default_rng(42)
    scatter = None

    for ax, model_key, display_label, model in axes:
        if model_key == "km_greedy":
            encoding_input = km_greedy_profiles
        elif model_key.startswith("joe_km_greedy_"):
            encoding_input = km_full_profiles
        else:
            encoding_input = ohe_encoder.transform(
                pd.DataFrame({
                    "cat_1": category_names,
                })
            ).astype("float32")

        encodings = np.asarray(
            model.encode(encoding_input)
        )

        if encodings.ndim == 1:
            encodings = encodings.reshape(-1, 1)

        if encodings.shape[0] != len(category_names):
            raise ValueError(
                f"Category/encoding mismatch for {model_key}: "
                f"{encodings.shape[0]} encoding rows but "
                f"{len(category_names)} categories."
            )

        if enc_dim == 1:
            x = encodings[:, 0]

            y = rng.uniform(
                -0.3,
                0.3,
                size=len(x),
            )

            scatter = ax.scatter(
                x,
                y,
                c=beta_g,
                cmap="viridis",
                norm=norm,
                s=55,
                alpha=0.85,
                edgecolors="white",
                linewidths=0.4,
            )

            ax.axhline(
                0,
                color="grey",
                linewidth=0.5,
                linestyle="--",
            )

            ax.set_xlabel(
                "Learned encoding",
                fontsize=11,
            )
            ax.set_yticks([])
            ax.set_ylabel("")

        elif enc_dim == 2:
            scatter = ax.scatter(
                encodings[:, 0],
                encodings[:, 1],
                c=beta_g,
                cmap="viridis",
                norm=norm,
                s=55,
                alpha=0.85,
                edgecolors="white",
                linewidths=0.4,
            )

            ax.set_xlabel("Dim 1", fontsize=11)
            ax.set_ylabel("Dim 2", fontsize=11)

        else:
            pca_enc = PCA(n_components=2)
            encodings_2d = pca_enc.fit_transform(encodings)

            explained_variance = (
                pca_enc.explained_variance_ratio_ * 100
            )

            scatter = ax.scatter(
                encodings_2d[:, 0],
                encodings_2d[:, 1],
                c=beta_g,
                cmap="viridis",
                norm=norm,
                s=55,
                alpha=0.85,
                edgecolors="white",
                linewidths=0.4,
            )

            ax.set_xlabel(
                f"Encoding PC1 "
                f"({explained_variance[0]:.1f}% var)",
                fontsize=11,
            )

            ax.set_ylabel(
                f"Encoding PC2 "
                f"({explained_variance[1]:.1f}% var)",
                fontsize=11,
            )

        ax.set_title(
            display_label,
            fontsize=11,
        )

    if scatter is not None:
        cax = fig.add_axes([
            1.05,
            0.18,
            0.025,
            0.64,
        ])

        colorbar = fig.colorbar(
            scatter,
            cax=cax,
        )

        colorbar.set_label(
            "True category effect",
            fontsize=10,
        )

    # fig.tight_layout(
    #     rect=[0, 0, 0.92, 1]
    # )

    output_path = os.path.join(
        out_dir,
        "encoding_noncluster.png",
    )

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(fig)

    print("  Saved encoding_noncluster.png")


def evaluate_embedding_structure(
    encodings: np.ndarray,
    true_effects: np.ndarray,
    true_cluster_labels: np.ndarray | None = None,
    random_state: int = 0,
) -> Dict[str, float]:
    """
    Quantitatively evaluate a learned category embedding.

    Metrics
    -------
    ARI
        K-means cluster recovery compared with the true cluster labels.
        Higher is better; 1 means perfect recovery.

    NMI
        Normalised mutual information between K-means labels and true labels.
        Higher is better; 1 means perfect agreement.

    Silhouette
        Separation of the known true clusters in embedding space.
        Higher is better. Values near or below zero indicate overlap.

    Effect-distance Spearman correlation
        Rank correlation between pairwise distances in embedding space and
        absolute differences in the true category effects.
        Higher is better; 1 means perfect rank preservation.
    """
    encodings = np.asarray(encodings, dtype=float)
    true_effects = np.asarray(true_effects, dtype=float)

    if encodings.ndim == 1:
        encodings = encodings.reshape(-1, 1)

    n_categories = encodings.shape[0]

    if len(true_effects) != n_categories:
        raise ValueError(
            "The encodings and true effects must contain "
            "the same number of categories."
        )

    if true_cluster_labels is not None:
        true_cluster_labels = np.asarray(true_cluster_labels)

        if len(true_cluster_labels) != n_categories:
            raise ValueError(
                "The encodings and true cluster labels must contain "
                "the same number of categories."
            )



    metrics = {
        "ari": float("nan"),
        "nmi": float("nan"),
        "silhouette_true": float("nan"),
        "effect_distance_spearman": float("nan"),
    }

    # --------------------------------------------------------------
    # 1. Recover clusters with K-means and compare with ground truth
    # --------------------------------------------------------------

    if true_cluster_labels is not None:
        unique_clusters = np.unique(true_cluster_labels)
        n_clusters = len(unique_clusters)

        if 1 < n_clusters < n_categories:
            kmeans = KMeans(
                n_clusters=n_clusters,
                n_init=50,
                random_state=random_state,
            )

            predicted_labels = kmeans.fit_predict(encodings)

            metrics["ari"] = float(
                adjusted_rand_score(
                    true_cluster_labels,
                    predicted_labels,
                )
            )

            metrics["nmi"] = float(
                normalized_mutual_info_score(
                    true_cluster_labels,
                    predicted_labels,
                )
            )

        # --------------------------------------------------------------
        # 2. Separation of the true clusters in embedding space
        # --------------------------------------------------------------
        cluster_sizes = np.array([
            np.sum(true_cluster_labels == cluster_id)
            for cluster_id in unique_clusters
        ])

        # Silhouette is only defined when there are at least two clusters,
        # fewer clusters than observations, and no singleton clusters.
        if (
            1 < n_clusters < n_categories
            and np.all(cluster_sizes >= 2)
        ):
            try:
                metrics["silhouette_true"] = float(
                    silhouette_score(
                        encodings,
                        true_cluster_labels,
                        metric="euclidean",
                    )
                )
            except ValueError:
                metrics["silhouette_true"] = float("nan")

    # --------------------------------------------------------------
    # 3. Preservation of the true category-effect geometry
    # --------------------------------------------------------------
    embedding_distances = pairwise_distances(
        encodings,
        metric="euclidean",
    )

    true_effect_distances = np.abs(
        true_effects[:, None] - true_effects[None, :]
    )

    # Use each unordered category pair exactly once.
    upper_triangle = np.triu_indices(n_categories, k=1)

    embedding_pair_distances = embedding_distances[upper_triangle]
    true_effect_pair_distances = true_effect_distances[upper_triangle]

    # Spearman correlation is undefined when either vector is constant.
    if (
        len(embedding_pair_distances) > 1
        and np.std(embedding_pair_distances) > 0
        and np.std(true_effect_pair_distances) > 0
    ):
        rho = spearmanr(
            embedding_pair_distances,
            true_effect_pair_distances,
        ).statistic

        metrics["effect_distance_spearman"] = float(rho)

    return metrics


def evaluate_all_category_encodings(
    trained_models: Dict[str, NeuralCox],
    km_greedy_profiles: np.ndarray,
    km_full_profiles: np.ndarray,
    km_category_names: List[str],
    ohe_encoder: OneHotEncoder,
    K: int,
    beta_g: np.ndarray,
    cluster_labels: np.ndarray | None,
    random_state: int,
) -> Dict[str, float]:
    """
    Evaluate all category representations and return flat result columns.

    The returned dictionary contains columns such as:

        ari_km_greedy
        nmi_joe_ohe_sigmoid
        silhouette_true_joe_km_greedy_linear
        effect_distance_spearman_joe_ohe_linear
    """
    all_category_names = category_labels(K)

    if cluster_labels is not None:
        label_to_cluster = {
            category_name: int(cluster_labels[i])
            for i, category_name in enumerate(all_category_names)
        }
    else:
        label_to_cluster = None

    label_to_effect = {
        category_name: float(beta_g[i])
        for i, category_name in enumerate(all_category_names)
    }

    output: Dict[str, float] = {}

    method_order = [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]

    for method_name in method_order:
        if method_name not in trained_models:
            continue

        model = trained_models[method_name]

        if method_name == "km_greedy":
            method_category_names = list(km_category_names)
            encoding_input = km_greedy_profiles
        elif method_name.startswith("joe_km_greedy_"):
            method_category_names = list(km_category_names)
            encoding_input = km_full_profiles
        else:
            method_category_names = all_category_names

            encoding_input = ohe_encoder.transform(
                pd.DataFrame({
                    "cat_1": method_category_names
                })
            ).astype("float32")

        # Retain only categories for which ground truth is available.
        valid_indices = [
            i
            for i, category_name in enumerate(method_category_names)
            if category_name in label_to_effect
        ]

        if len(valid_indices) < 3:
            print(
                f"  Warning: not enough valid categories to evaluate "
                f"{method_name}."
            )
            continue

        encoding_input_valid = encoding_input[valid_indices]

        valid_category_names = [
            method_category_names[i]
            for i in valid_indices
        ]

        if label_to_cluster is not None:
            method_true_clusters = np.array([
                label_to_cluster[name]
                for name in valid_category_names
            ])
        else:
            method_true_clusters = None



        method_true_effects = np.array([
            label_to_effect[name]
            for name in valid_category_names
        ])

        method_encodings = model.encode(encoding_input_valid)

        method_metrics = evaluate_embedding_structure(
            encodings=method_encodings,
            true_effects=method_true_effects,
            true_cluster_labels=method_true_clusters,
            random_state=random_state,
        )

        for metric_name, metric_value in method_metrics.items():
            output[f"{metric_name}_{method_name}"] = metric_value

        if method_true_clusters is not None:
            print(
                f"  Structure {method_name}: "
                f"ARI={method_metrics['ari']:.3f}, "
                f"NMI={method_metrics['nmi']:.3f}, "
                f"silhouette={method_metrics['silhouette_true']:.3f}, "
                f"effect-distance rho="
                f"{method_metrics['effect_distance_spearman']:.3f}"
            )
        else:
            print(
                f"  Structure {method_name}: "
                f"effect-distance rho="
                f"{method_metrics['effect_distance_spearman']:.3f}"
            )

    return output






@dataclass
class ExperimentConfig:
    n_train: int = 500
    n_test: int = 5000
    K: int = 40
    censor_prop: float = 0.4
    encoding_dim: int = ENCODING_DIM
    epochs: int = EPOCHS
    lr: float = LR


def run_one_experiment(
    cfg: ExperimentConfig,
    beta_g: np.ndarray,
    cluster_labels: np.ndarray | None,
    seed: int,
    return_models: bool = False,
    out_dir: str = "results_ch5_comparison",
) -> Dict:
    train_df = generate_survival_dataset(
        cfg.n_train,
        cfg.K,
        beta_g,
        censor_prop=cfg.censor_prop,
    )

    test_df = generate_survival_dataset(
        cfg.n_test,
        cfg.K,
        beta_g,
        censor_prop=cfg.censor_prop,
    )

    time_train = train_df["Time"].to_numpy()
    event_train = train_df["Event"].to_numpy()
    time_test = test_df["Time"].to_numpy()
    event_test = test_df["Event"].to_numpy()

    X_cont_train, X_cont_test = build_cont_arrays(train_df, test_df)
    X_ohe_train, X_ohe_test, ohe_encoder = build_ohe_arrays(train_df, test_df)

    r_work_dir = os.path.join(
        out_dir,
        "r_km_greedy_cache",
        f"K_{cfg.K}_censor_{cfg.censor_prop}_seed_{seed}",
    )

    (
        X_km_greedy_train,
        X_km_greedy_test,
        X_km_greedy_cat_profiles,
        X_km_full_train,
        X_km_full_test,
        X_km_full_cat_profiles,
        km_cat_labels,
    ) = run_r_km_encoder(
        train_df=train_df,
        test_df=test_df,
        work_dir=r_work_dir,
    )

    km_greedy_dim = X_km_greedy_train.shape[1]
    km_full_dim = X_km_full_train.shape[1]
    ohe_dim = X_ohe_train.shape[1]
    enc_dim = cfg.encoding_dim

    results: Dict = {
        "n_train": cfg.n_train,
        "K": cfg.K,
        "censor_prop": cfg.censor_prop,
        "encoding_dim": enc_dim,
        "events_train": int(train_df["Event"].sum()),
        "km_greedy_dim": int(km_greedy_dim),
        "km_full_dim": int(km_full_dim),
    }

    trained_models: Dict[str, NeuralCox] = {}

    print("NN OHE:")
    model_nn = NeuralCox(
        encoder=None,
        cat_input_dim=ohe_dim,
        cont_dim=2,
        name="nn_ohe",
    )

    train_neural_cox(
        model_nn,
        X_ohe_train,
        X_cont_train,
        time_train,
        event_train,
        epochs=cfg.epochs,
        lr=cfg.lr,
    )

    scores = score_model_sksurv(
        model_nn,
        X_ohe_train,
        X_cont_train,
        X_ohe_test,
        X_cont_test,
        time_train,
        event_train,
        time_test,
        event_test,
    )

    results["cindex_nn_ohe"] = scores["cindex"]
    results["ibs_nn_ohe"] = scores["ibs"]
    trained_models["nn_ohe"] = model_nn

    for variant, hidden_units, last_act in [
        ("sigmoid", [3, enc_dim], "sigmoid"),
        ("linear", [enc_dim], "linear"),
    ]:
        print(f"JOE KM-FULL {hidden_units} {variant}")

        enc = SmallEncoder(
            hidden_units,
            last_activation=last_act,
            name=f"km_full_enc_{variant}",
        )

        model = NeuralCox(
            encoder=enc,
            cat_input_dim=km_full_dim,
            cont_dim=2,
            name=f"joe_km_greedy_{variant}",
        )

        train_neural_cox(
            model,
            X_km_full_train,
            X_cont_train,
            time_train,
            event_train,
            epochs=cfg.epochs,
            lr=cfg.lr,
        )

        key = f"joe_km_greedy_{variant}"

        scores = score_model_sksurv(
            model,
            X_km_full_train,
            X_cont_train,
            X_km_full_test,
            X_cont_test,
            time_train,
            event_train,
            time_test,
            event_test,
        )

        results[f"cindex_{key}"] = scores["cindex"]
        results[f"ibs_{key}"] = scores["ibs"]
        trained_models[key] = model

    for variant, hidden_units, last_act in [
        ("sigmoid", [3, enc_dim], "sigmoid"),
        ("linear", [enc_dim], "linear"),
    ]:
        print(f"JOE OHE {hidden_units} {variant}")

        enc = SmallEncoder(
            hidden_units,
            last_activation=last_act,
            name=f"ohe_enc_{variant}",
        )

        model = NeuralCox(
            encoder=enc,
            cat_input_dim=ohe_dim,
            cont_dim=2,
            name=f"joe_ohe_{variant}",
        )

        train_neural_cox(
            model,
            X_ohe_train,
            X_cont_train,
            time_train,
            event_train,
            epochs=cfg.epochs,
            lr=cfg.lr,
        )

        key = f"joe_ohe_{variant}"

        scores = score_model_sksurv(
            model,
            X_ohe_train,
            X_cont_train,
            X_ohe_test,
            X_cont_test,
            time_train,
            event_train,
            time_test,
            event_test,
        )

        results[f"cindex_{key}"] = scores["cindex"]
        results[f"ibs_{key}"] = scores["ibs"]
        trained_models[key] = model

    base_ci = results["cindex_nn_ohe"]


    print("KM-GREEDY:")

    model_km = NeuralCox(
        encoder=None,
        cat_input_dim=km_greedy_dim,
        cont_dim=2,
        name="km_greedy",
    )

    train_neural_cox(
        model_km,
        X_km_greedy_train,
        X_cont_train,
        time_train,
        event_train,
        epochs=cfg.epochs,
        lr=cfg.lr,
    )

    scores = score_model_sksurv(
        model_km,
        X_km_greedy_train,
        X_cont_train,
        X_km_greedy_test,
        X_cont_test,
        time_train,
        event_train,
        time_test,
        event_test,
    )

    results["cindex_km_greedy"] = scores["cindex"]
    results["ibs_km_greedy"] = scores["ibs"]
    trained_models["km_greedy"] = model_km


    # --------------------------------------------------------------
    # Quantitative evaluation of category embedding structure
    # --------------------------------------------------------------
    structure_metrics = evaluate_all_category_encodings(
        trained_models=trained_models,
        km_greedy_profiles=X_km_greedy_cat_profiles,
        km_full_profiles=X_km_full_cat_profiles,
        km_category_names=km_cat_labels,
        ohe_encoder=ohe_encoder,
        K=cfg.K,
        beta_g=beta_g,
        cluster_labels=cluster_labels,
        random_state=seed,
    )

    results.update(structure_metrics)




    base_ibs = results["ibs_nn_ohe"]

    for key in [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]:
        # C-index: higher is better, so gain is method - NN OHE baseline.
        results[f"gain_cindex_{key}"] = results[f"cindex_{key}"] - base_ci

        # IBS: lower is better, so define improvement as baseline - method.
        # Positive values therefore mean better than NN OHE, matching C-index gains.
        results[f"gain_ibs_{key}"] = base_ibs - results[f"ibs_{key}"]

    if return_models:
        results["_trained_models"] = trained_models
        results["_X_km_greedy_cat_profiles"] = X_km_greedy_cat_profiles
        results["_X_km_full_cat_profiles"] = X_km_full_cat_profiles
        results["_km_cat_labels"] = km_cat_labels
        results["_ohe_encoder"] = ohe_encoder
        results["_train_df"] = train_df

    return results


def run_grid(
    K_grid,
    censor_grid,
    seeds,
    out_dir: str,
    experiment_type: str,
) -> pd.DataFrame:
    rows = []

    for K in K_grid:
        for censor in censor_grid:
            for seed in seeds:
                print(
                    f"  experiment={experiment_type}  "
                    f"K={K}  censor={censor}  seed={seed}"
                )

                np.random.seed(seed)
                tf.random.set_seed(seed)

                beta_g, cluster_labels = make_experiment_effects(
                    K=K,
                    experiment_type=experiment_type,
                )

                cfg = ExperimentConfig(K=K, censor_prop=censor)

                try:
                    res = run_one_experiment(
                        cfg,
                        beta_g,
                        cluster_labels,
                        seed=seed,
                        return_models=True,
                        out_dir=out_dir,
                    )

                    res["experiment_type"] = experiment_type
                    res["seed"] = seed

                    trained_models = res.pop("_trained_models")
                    km_greedy_cat_profiles = res.pop("_X_km_greedy_cat_profiles")
                    km_full_cat_profiles = res.pop("_X_km_full_cat_profiles")
                    km_cat_labels = res.pop("_km_cat_labels")
                    ohe_encoder = res.pop("_ohe_encoder")
                    train_df_for_plot = res.pop("_train_df")

                    rows.append(res)

                    vis_dir = os.path.join(
                        out_dir,
                        "encoding_visualisations",
                        f"K_{K}_censor_{censor}_seed_{seed}",
                    )
                    os.makedirs(vis_dir, exist_ok=True)

                    if experiment_type == "cluster":

                        plot_encoding_diagnostics_cluster(
                            K=K,
                            cluster_labels=cluster_labels,
                            trained_models=trained_models,
                            km_greedy_profiles=km_greedy_cat_profiles,
                            km_full_profiles=km_full_cat_profiles,
                            enc_dim=cfg.encoding_dim,
                            out_dir=vis_dir,
                            ohe_encoder=ohe_encoder,
                            train_df=train_df_for_plot,
                            km_cat_labels=km_cat_labels,
                        )

                    else:
                        label_to_beta = {
                            category_labels(K)[i]: float(beta_g[i])
                            for i in range(K)
                        }
                        km_plot_beta_g = np.array([
                            label_to_beta[label]
                            for label in km_cat_labels
                        ])

                        plot_encoding_diagnostics_noncluster(
                            beta_g=km_plot_beta_g,
                            trained_models=trained_models,
                            km_greedy_profiles=km_greedy_cat_profiles,
                            km_full_profiles=km_full_cat_profiles,
                            enc_dim=cfg.encoding_dim,
                            out_dir=vis_dir,
                            ohe_encoder=ohe_encoder,
                            category_names=km_cat_labels,
                        )

                except Exception as exc:
                    rows.append({
                        "experiment_type": experiment_type,
                        "K": K,
                        "censor_prop": censor,
                        "seed": seed,
                        "km_greedy_dim": None,
                        "error": str(exc),
                    })

    return pd.DataFrame(rows)


# def plot_metric_gain_boxplots(
#     df_res: pd.DataFrame,
#     out_dir: str,
# ) -> None:
#     """Save side-by-side seed-level gain plots for C-index and IBS.

#     C-index gain is defined as method C-index minus NN OHE C-index.
#     IBS gain is defined as NN OHE IBS minus method IBS, because lower IBS is better.
#     Therefore, positive values are better than NN OHE for both plots.
#     """
#     import matplotlib.pyplot as plt

#     method_order = [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]

#     method_labels = {
#         "km_greedy": "KM",
#         "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
#         "joe_km_greedy_linear": "JOECHAR_1LIN",
#         "joe_ohe_sigmoid": "JOEOH_3SIG",
#         "joe_ohe_linear": "JOEOH_1LIN",
#     }

#     plot_specs = [
#         (
#             "gain_cindex",
#             "C-index gain vs NN OHE",
#             "C-index gain\n(method - NN OHE)",
#             "cindex_gain_boxplot.png",
#         ),
#         (
#             "gain_ibs",
#             "IBS improvement vs NN OHE",
#             "IBS improvement\n(NN OHE - method)",
#             "ibs_gain_boxplot.png",
#         ),
#     ]

#     for prefix, title, ylabel, filename in plot_specs:
#         cols = [f"{prefix}_{m}" for m in method_order if f"{prefix}_{m}" in df_res.columns]
#         labels = [method_labels[m] for m in method_order if f"{prefix}_{m}" in df_res.columns]

#         if not cols:
#             print(f"  No columns found for {prefix}; skipping gain plot.")
#             continue

#         data = [df_res[c].dropna().to_numpy() for c in cols]
#         keep = [i for i, arr in enumerate(data) if len(arr) > 0]

#         if not keep:
#             print(f"  No non-missing data found for {prefix}; skipping gain plot.")
#             continue

#         data = [data[i] for i in keep]
#         labels = [labels[i] for i in keep]

#         fig, ax = plt.subplots(figsize=(1.6 * len(data) + 3, 4.8))
#         bp = ax.boxplot(
#             data,
#             labels=labels,
#             showmeans=False,
#             patch_artist=True,
#         )

#         for box in bp["boxes"]:
#             box.set_alpha(0.65)

#         ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
#         ax.set_title(title, fontsize=12)
#         ax.set_ylabel(ylabel, fontsize=11)
#         ax.tick_params(axis="x", labelrotation=30)
#         # ax.grid(axis="y", alpha=0.25)

#         fig.tight_layout()
#         fig.savefig(os.path.join(out_dir, filename), dpi=150, bbox_inches="tight")
#         plt.close(fig)

#         print(f"  Saved {filename}")

# def plot_embedding_structure_boxplots(
#     df_res: pd.DataFrame,
#     out_dir: str,
# ) -> None:
#     """
#     Plot seed-level distributions of embedding-structure metrics.
#     """
#     import matplotlib.pyplot as plt

#     method_order = [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]

#     method_labels = {
#         "km_greedy": "KM-GREEDY",
#         "joe_km_greedy_sigmoid": "JOE KM sigmoid",
#         "joe_km_greedy_linear": "JOE KM linear",
#         "joe_ohe_sigmoid": "JOE OHE sigmoid",
#         "joe_ohe_linear": "JOE OHE linear",
#     }

#     plot_specs = [
#         (
#             "ari",
#             "Adjusted Rand index",
#             "ARI",
#             "embedding_ari_boxplot.png",
#             0.0,
#             1.0,
#         ),
#         (
#             "nmi",
#             "Normalised mutual information",
#             "NMI",
#             "embedding_nmi_boxplot.png",
#             0.0,
#             1.0,
#         ),
#         (
#             "silhouette_true",
#             "True-cluster silhouette score",
#             "Silhouette score",
#             "embedding_silhouette_boxplot.png",
#             -1.0,
#             1.0,
#         ),
#         (
#             "effect_distance_spearman",
#             "Preservation of true effect distances",
#             "Spearman correlation",
#             "embedding_effect_distance_boxplot.png",
#             -1.0,
#             1.0,
#         ),
#     ]

#     for (
#         metric_prefix,
#         title,
#         ylabel,
#         filename,
#         lower_limit,
#         upper_limit,
#     ) in plot_specs:

#         columns = [
#             f"{metric_prefix}_{method}"
#             for method in method_order
#             if f"{metric_prefix}_{method}" in df_res.columns
#         ]

#         labels = [
#             method_labels[method]
#             for method in method_order
#             if f"{metric_prefix}_{method}" in df_res.columns
#         ]

#         if not columns:
#             print(
#                 f"  No columns found for {metric_prefix}; "
#                 f"skipping plot."
#             )
#             continue

#         data = [
#             df_res[column]
#             .replace([np.inf, -np.inf], np.nan)
#             .dropna()
#             .to_numpy()
#             for column in columns
#         ]

#         keep_indices = [
#             i
#             for i, values in enumerate(data)
#             if len(values) > 0
#         ]

#         if not keep_indices:
#             print(
#                 f"  No valid values found for {metric_prefix}; "
#                 f"skipping plot."
#             )
#             continue

#         data = [data[i] for i in keep_indices]
#         labels = [labels[i] for i in keep_indices]

#         fig, ax = plt.subplots(
#             figsize=(1.7 * len(data) + 3.0, 4.8)
#         )

#         boxplot = ax.boxplot(
#             data,
#             labels=labels,
#             showmeans=False,
#             patch_artist=True,
#         )

#         for box in boxplot["boxes"]:
#             box.set_alpha(0.65)

#         # Add individual seed results.
#         rng = np.random.default_rng(42)

#         for position, values in enumerate(data, start=1):
#             jitter = rng.normal(
#                 loc=0.0,
#                 scale=0.035,
#                 size=len(values),
#             )

#             ax.scatter(
#                 np.full(len(values), position) + jitter,
#                 values,
#                 s=25,
#                 alpha=0.65,
#                 zorder=3,
#             )

#         ax.set_title(title, fontsize=12)
#         ax.set_ylabel(ylabel, fontsize=11)
#         ax.set_ylim(lower_limit, upper_limit)
#         ax.tick_params(axis="x", labelrotation=30)
#         # ax.grid(axis="y", alpha=0.25)

#         fig.tight_layout()
#         fig.savefig(
#             os.path.join(out_dir, filename),
#             dpi=150,
#             bbox_inches="tight",
#         )
#         plt.close(fig)

#         print(f"  Saved {filename}")


if __name__ == "__main__":
    # Choose exactly one: "cluster" or "noncluster"
    EXPERIMENT_TYPE = "cluster"

    out_dir = f"results_ch5_comparison_{EXPERIMENT_TYPE}"
    os.makedirs(out_dir, exist_ok=True)

    K_grid = [40]
    censor_grid = [0.3]
    seeds = range(50)

    df_res = run_grid(
        K_grid=K_grid,
        censor_grid=censor_grid,
        seeds=seeds,
        out_dir=out_dir,
        experiment_type=EXPERIMENT_TYPE,
    )

    df_res.to_csv(
        os.path.join(out_dir, "chapter5_comparison_results.csv"),
        index=False,
    )

    # plot_metric_gain_boxplots(df_res=df_res, out_dir=out_dir)
    # plot_embedding_structure_boxplots(
    #     df_res=df_res,
    #     out_dir=out_dir,
    # )

    method_cols = [
        "cindex_nn_ohe",
        "cindex_km_greedy",
        "cindex_joe_km_greedy_sigmoid",
        "cindex_joe_km_greedy_linear",
        "cindex_joe_ohe_sigmoid",
        "cindex_joe_ohe_linear",
    ]

    gain_cindex_cols = [
        "gain_cindex_km_greedy",
        "gain_cindex_joe_km_greedy_sigmoid",
        "gain_cindex_joe_km_greedy_linear",
        "gain_cindex_joe_ohe_sigmoid",
        "gain_cindex_joe_ohe_linear",
    ]

    gain_ibs_cols = [
        "gain_ibs_km_greedy",
        "gain_ibs_joe_km_greedy_sigmoid",
        "gain_ibs_joe_km_greedy_linear",
        "gain_ibs_joe_ohe_sigmoid",
        "gain_ibs_joe_ohe_linear",
    ]

    gain_cols = gain_cindex_cols + gain_ibs_cols

    ibs_cols = [
        "ibs_nn_ohe",
        "ibs_km_greedy",
        "ibs_joe_km_greedy_sigmoid",
        "ibs_joe_km_greedy_linear",
        "ibs_joe_ohe_sigmoid",
        "ibs_joe_ohe_linear",
    ]

    ok_cols = [c for c in method_cols if c in df_res.columns]

    structure_metric_prefixes = [
        "ari_",
        "nmi_",
        "silhouette_true_",
        "effect_distance_spearman_",
    ]

    structure_cols = [
        column
        for column in df_res.columns
        if any(
            column.startswith(prefix)
            for prefix in structure_metric_prefixes
        )
    ]

    summary = (
        df_res.dropna(subset=ok_cols)
        .groupby(
            ["experiment_type", "K", "censor_prop"],
            as_index=False,
        )
        .agg(
            **{
                f"mean_{column}": (column, "mean")
                for column in method_cols
                if column in df_res.columns
            },
            **{
                f"sd_{column}": (column, "std")
                for column in method_cols
                if column in df_res.columns
            },
            **{
                f"mean_{column}": (column, "mean")
                for column in gain_cols
                if column in df_res.columns
            },
            **{
                f"mean_{column}": (column, "mean")
                for column in ibs_cols
                if column in df_res.columns
            },
            **{
                f"mean_{column}": (column, "mean")
                for column in structure_cols
            },
            **{
                f"sd_{column}": (column, "std")
                for column in structure_cols
            },
            **(
                {}
                if "km_greedy_dim" not in df_res.columns
                else {
                    "mean_km_greedy_dim": (
                        "km_greedy_dim",
                        "mean",
                    )
                }
            ),
        )
    )

    summary.to_csv(
        os.path.join(out_dir, "chapter5_comparison_summary.csv"),
        index=False,
    )

    print("\n=== Mean C-index by method ===")
    print(summary.to_string(index=False))

    print(f"\nAll outputs saved to '{out_dir}/'")

  experiment=cluster  K=40  censor=0.3  seed=0
NN OHE:
  epoch=0001  loss=5.58774  lr=0.1000000
  epoch=0050  loss=4.80339  lr=0.1000000
  epoch=0100  loss=4.79206  lr=0.1000000
  epoch=0150  loss=4.79121  lr=0.1000000

Epoch 157: ReduceLROnPlateau reducing learning rate to 0.05000000074505806.

Epoch 177: ReduceLROnPlateau reducing learning rate to 0.02500000037252903.

Epoch 197: ReduceLROnPlateau reducing learning rate to 0.012500000186264515.
  epoch=0200  loss=4.79117  lr=0.0125000

Epoch 217: ReduceLROnPlateau reducing learning rate to 0.0062500000931322575.

Epoch 237: ReduceLROnPlateau reducing learning rate to 0.0031250000465661287.
  epoch=0250  loss=4.79117  lr=0.0031250

Epoch 257: ReduceLROnPlateau reducing learning rate to 0.0015625000232830644.

Epoch 277: ReduceLROnPlateau reducing learning rate to 0.0007812500116415322.

Epoch 297: ReduceLROnPlateau reducing learning rate to 0.0003906250058207661.
  epoch=0300  loss=4.79117  lr=0.0003906

Epoch 317: ReduceLROnPlateau r

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


NN OHE:
  epoch=0001  loss=5.38945  lr=0.1000000
  epoch=0050  loss=4.68190  lr=0.1000000
  epoch=0100  loss=4.65712  lr=0.1000000
  epoch=0150  loss=4.64953  lr=0.1000000
  epoch=0200  loss=4.64651  lr=0.1000000
  epoch=0250  loss=4.64514  lr=0.1000000
  epoch=0300  loss=4.64449  lr=0.1000000
  epoch=0350  loss=4.64415  lr=0.1000000

Epoch 358: ReduceLROnPlateau reducing learning rate to 0.05000000074505806.

Epoch 379: ReduceLROnPlateau reducing learning rate to 0.02500000037252903.

Epoch 399: ReduceLROnPlateau reducing learning rate to 0.012500000186264515.
  epoch=0400  loss=4.64406  lr=0.0125000
JOE KM-FULL [3, 2] sigmoid
  epoch=0001  loss=5.52204  lr=0.1000000
  epoch=0050  loss=4.68903  lr=0.1000000
  epoch=0100  loss=4.67386  lr=0.1000000
  epoch=0150  loss=4.66797  lr=0.1000000
  epoch=0200  loss=4.66662  lr=0.1000000

Epoch 235: ReduceLROnPlateau reducing learning rate to 0.05000000074505806.
  epoch=0250  loss=4.66563  lr=0.0500000
  epoch=0300  loss=4.66477  lr=0.0500000


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


  Structure joe_ohe_sigmoid: ARI=0.569, NMI=0.683, silhouette=0.342, effect-distance rho=0.656
  Structure joe_ohe_linear: ARI=0.498, NMI=0.653, silhouette=0.243, effect-distance rho=0.690
  Saved encoding_clusters.png
  experiment=cluster  K=40  censor=0.3  seed=2


...

  Saved encoding_clusters.png
  experiment=cluster  K=40  censor=0.3  seed=48


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


NN OHE:
  epoch=0001  loss=5.45891  lr=0.1000000
  epoch=0050  loss=4.91027  lr=0.1000000
  epoch=0100  loss=4.89775  lr=0.1000000
  epoch=0150  loss=4.89708  lr=0.1000000

Epoch 159: ReduceLROnPlateau reducing learning rate to 0.05000000074505806.

Epoch 179: ReduceLROnPlateau reducing learning rate to 0.02500000037252903.

Epoch 199: ReduceLROnPlateau reducing learning rate to 0.012500000186264515.
  epoch=0200  loss=4.89706  lr=0.0125000

Epoch 219: ReduceLROnPlateau reducing learning rate to 0.0062500000931322575.

Epoch 239: ReduceLROnPlateau reducing learning rate to 0.0031250000465661287.
  epoch=0250  loss=4.89706  lr=0.0031250

Epoch 259: ReduceLROnPlateau reducing learning rate to 0.0015625000232830644.

Epoch 279: ReduceLROnPlateau reducing learning rate to 0.0007812500116415322.

Epoch 299: ReduceLROnPlateau reducing learning rate to 0.0003906250058207661.
  epoch=0300  loss=4.89706  lr=0.0003906

Epoch 319: ReduceLROnPlateau reducing learning rate to 0.00019531250291038305

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


  Structure joe_ohe_linear: ARI=0.405, NMI=0.516, silhouette=0.146, effect-distance rho=0.697
  Saved encoding_clusters.png
  experiment=cluster  K=40  censor=0.3  seed=49
NN OHE:
  epoch=0001  loss=5.71624  lr=0.1000000
  epoch=0050  loss=4.79336  lr=0.1000000
  epoch=0100  loss=4.78377  lr=0.1000000
  epoch=0150  loss=4.78224  lr=0.1000000
  epoch=0200  loss=4.78188  lr=0.1000000

Epoch 204: ReduceLROnPlateau reducing learning rate to 0.05000000074505806.

Epoch 224: ReduceLROnPlateau reducing learning rate to 0.02500000037252903.

Epoch 246: ReduceLROnPlateau reducing learning rate to 0.012500000186264515.
  epoch=0250  loss=4.78183  lr=0.0125000

Epoch 266: ReduceLROnPlateau reducing learning rate to 0.0062500000931322575.

Epoch 286: ReduceLROnPlateau reducing learning rate to 0.0031250000465661287.
  epoch=0300  loss=4.78183  lr=0.0031250

Epoch 306: ReduceLROnPlateau reducing learning rate to 0.0015625000232830644.

Epoch 326: ReduceLROnPlateau reducing learning rate to 0.000781

In [11]:
# def plot_metric_gain_boxplots(
#     df_res: pd.DataFrame,
#     out_dir: str,
#     k_col: str = "K",
# ) -> None:
#     """Plot gain distributions grouped by K.

#     Within each K group, one boxplot is drawn for every available method.
#     Vertical dashed lines separate consecutive K groups.

#     C-index gain:
#         method C-index - NN OHE C-index

#     IBS gain:
#         NN OHE IBS - method IBS

#     Positive values therefore indicate improvement over NN OHE.
#     """
#     import os
#     import numpy as np
#     import matplotlib.pyplot as plt

#     method_order = [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]

#     method_labels = {
#         "km_greedy": "KM",
#         "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
#         "joe_km_greedy_linear": "JOECHAR_1LIN",
#         "joe_ohe_sigmoid": "JOEOH_3SIG",
#         "joe_ohe_linear": "JOEOH_1LIN",
#     }

#     plot_specs = [
#         (
#             "gain_cindex",
#             "C-index gain\n(method - NN OHE)",
#             "cindex_gain_boxplot.png",
#         ),
#         (
#             "gain_ibs",
#             "IBS gain\n(NN OHE - method)",
#             "ibs_gain_boxplot.png",
#         ),
#     ]

#     if k_col not in df_res.columns:
#         raise KeyError(
#             f"Column {k_col!r} was not found in df_res. "
#             f"Available columns: {list(df_res.columns)}"
#         )

#     # Sorting gives groups such as K=10, K=20, K=40.
#     k_values = sorted(df_res[k_col].dropna().unique())

#     if not k_values:
#         print("  No K values found; skipping gain plots.")
#         return

#     for prefix, ylabel, filename in plot_specs:
#         all_data = []
#         positions = []
#         tick_labels = []

#         group_centres = []
#         group_labels = []
#         separator_positions = []

#         current_position = 1

#         for group_index, k_value in enumerate(k_values):
#             df_k = df_res.loc[df_res[k_col] == k_value]

#             group_positions = []

#             for method in method_order:
#                 column = f"{prefix}_{method}"

#                 if column not in df_k.columns:
#                     continue

#                 values = (
#                     df_k[column]
#                     .replace([np.inf, -np.inf], np.nan)
#                     .dropna()
#                     .to_numpy()
#                 )

#                 if len(values) == 0:
#                     continue

#                 all_data.append(values)
#                 positions.append(current_position)
#                 tick_labels.append(method_labels[method])
#                 group_positions.append(current_position)

#                 current_position += 1

#             # Record the centre of this K group for the upper group label.
#             if group_positions:
#                 group_centres.append(np.mean(group_positions))
#                 group_labels.append(f"K = {k_value}")

#                 # Separator goes midway between this group and the next.
#                 separator_positions.append(current_position - 0.5)

#                 # Add a small blank space between K groups.
#                 current_position += 1

#         if not all_data:
#             print(f"  No valid data found for {prefix}; skipping gain plot.")
#             continue

#         # The final separator would appear after the last group, so remove it.
#         separator_positions = separator_positions[:-1]

#         figure_width = max(10, 0.9 * len(all_data) + 2.5)

#         fig, ax = plt.subplots(
#             figsize=(figure_width, 5.2)
#         )

#         boxplot = ax.boxplot(
#             all_data,
#             positions=positions,
#             widths=0.65,
#             patch_artist=True,
#             showmeans=False,
#         )

#         for box in boxplot["boxes"]:
#             box.set(
#                 facecolor="lightgrey",
#                 edgecolor="black",
#                 alpha=1.0,
#             )

#         for whisker in boxplot["whiskers"]:
#             whisker.set(color="black")

#         for cap in boxplot["caps"]:
#             cap.set(color="black")

#         for median in boxplot["medians"]:
#             median.set(
#                 color="black",
#                 linewidth=2,
#             )

#         for flier in boxplot["fliers"]:
#             flier.set(
#                 marker="o",
#                 markerfacecolor="none",
#                 markeredgecolor="black",
#                 markersize=4,
#             )

#         # Baseline: no difference from NN OHE.
#         ax.axhline(
#             0,
#             color="grey",
#             linewidth=0.8,
#             linestyle="--",
#             zorder=0,
#         )

#         # Separate the K groups.
#         for separator in separator_positions:
#             ax.axvline(
#                 separator,
#                 color="grey",
#                 linewidth=0.8,
#                 linestyle="--",
#                 alpha=0.8,
#                 zorder=0,
#             )

#         ax.set_xticks(positions)
#         ax.set_xticklabels(
#             tick_labels,
#             rotation=30,
#             ha="right",
#         )

#         ax.set_ylabel(ylabel, fontsize=11)

#         # Add a second x-axis above the plot containing the K labels.
#         top_axis = ax.secondary_xaxis("top")
#         top_axis.set_xticks(group_centres)
#         top_axis.set_xticklabels(group_labels)
#         top_axis.tick_params(
#             axis="x",
#             length=0,
#             pad=8,
#             labelsize=11,
#         )

#         # Hide the secondary-axis border.
#         top_axis.spines["top"].set_visible(False)

#         ax.set_xlim(
#             min(positions) - 0.7,
#             max(positions) + 0.7,
#         )

#         fig.tight_layout()
#         fig.savefig(
#             os.path.join(out_dir, filename),
#             dpi=150,
#             bbox_inches="tight",
#         )
#         plt.close(fig)

#         print(f"  Saved {filename}")

# def plot_effect_distance_boxplot_by_k(
#     df_res: pd.DataFrame,
#     out_dir: str,
#     k_col: str = "K",
# ) -> None:
#     """
#     Plot effect-distance Spearman correlation distributions grouped by K.

#     Within every K group, one boxplot is shown for each embedding method.
#     Thus results from different K values are NOT pooled together.
#     """
#     import os
#     import numpy as np
#     import matplotlib.pyplot as plt

#     method_order = [
#         "km_greedy",
#         "joe_km_greedy_sigmoid",
#         "joe_km_greedy_linear",
#         "joe_ohe_sigmoid",
#         "joe_ohe_linear",
#     ]

#     method_labels = {
#         "km_greedy": "KM",
#         "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
#         "joe_km_greedy_linear": "JOECHAR_1LIN",
#         "joe_ohe_sigmoid": "JOEOH_3SIG",
#         "joe_ohe_linear": "JOEOH_1LIN",
#     }

#     metric_prefix = "effect_distance_spearman"

#     if k_col not in df_res.columns:
#         raise KeyError(
#             f"Column {k_col!r} not found in df_res."
#         )

#     k_values = sorted(
#         df_res[k_col].dropna().unique()
#     )

#     if not k_values:
#         print("  No K values found; skipping distance plot.")
#         return

#     all_data = []
#     positions = []
#     tick_labels = []

#     group_centres = []
#     group_labels = []
#     separator_positions = []

#     current_position = 1

#     # ----------------------------------------------------------
#     # Treat every K separately
#     # ----------------------------------------------------------
#     for k_value in k_values:

#         df_k = df_res.loc[
#             df_res[k_col] == k_value
#         ]

#         group_positions = []

#         for method in method_order:

#             column = f"{metric_prefix}_{method}"

#             if column not in df_k.columns:
#                 continue

#             values = (
#                 df_k[column]
#                 .replace([np.inf, -np.inf], np.nan)
#                 .dropna()
#                 .to_numpy()
#             )

#             if len(values) == 0:
#                 continue

#             all_data.append(values)
#             positions.append(current_position)
#             tick_labels.append(
#                 method_labels[method]
#             )

#             group_positions.append(
#                 current_position
#             )

#             current_position += 1

#         # Centre label for this K group
#         if group_positions:

#             group_centres.append(
#                 np.mean(group_positions)
#             )

#             group_labels.append(
#                 f"K = {k_value}"
#             )

#             separator_positions.append(
#                 current_position - 0.5
#             )

#             # blank space between K groups
#             current_position += 1

#     if not all_data:
#         print(
#             "  No valid effect-distance values found; "
#             "skipping plot."
#         )
#         return

#     # Do not draw separator after final K group
#     separator_positions = separator_positions[:-1]

#     # ----------------------------------------------------------
#     # Plot
#     # ----------------------------------------------------------
#     figure_width = max(
#         10,
#         0.9 * len(all_data) + 2.5
#     )

#     fig, ax = plt.subplots(
#         figsize=(figure_width, 5.2)
#     )

#     boxplot = ax.boxplot(
#         all_data,
#         positions=positions,
#         widths=0.65,
#         patch_artist=True,
#         showmeans=False,
#     )

#     # ----------------------------------------------------------
#     # Styling -- same as C-index / IBS plots
#     # ----------------------------------------------------------
#     for box in boxplot["boxes"]:
#         box.set(
#             facecolor="lightgrey",
#             edgecolor="black",
#             alpha=1.0,
#         )

#     for whisker in boxplot["whiskers"]:
#         whisker.set(color="black")

#     for cap in boxplot["caps"]:
#         cap.set(color="black")

#     for median in boxplot["medians"]:
#         median.set(
#             color="black",
#             linewidth=2,
#         )

#     for flier in boxplot["fliers"]:
#         flier.set(
#             marker="o",
#             markerfacecolor="none",
#             markeredgecolor="black",
#             markersize=4,
#         )

#     # ----------------------------------------------------------
#     # Separate K groups
#     # ----------------------------------------------------------
#     for separator in separator_positions:
#         ax.axvline(
#             separator,
#             color="grey",
#             linewidth=0.8,
#             linestyle="--",
#             alpha=0.8,
#             zorder=0,
#         )

#     # ----------------------------------------------------------
#     # Method labels
#     # ----------------------------------------------------------
#     ax.set_xticks(positions)

#     ax.set_xticklabels(
#         tick_labels,
#         rotation=30,
#         ha="right",
#     )

#     ax.set_ylabel(
#         "Effect-distance Spearman correlation",
#         fontsize=11,
#     )

#     # ----------------------------------------------------------
#     # K labels on top
#     # ----------------------------------------------------------
#     top_axis = ax.secondary_xaxis("top")

#     top_axis.set_xticks(
#         group_centres
#     )

#     top_axis.set_xticklabels(
#         group_labels
#     )

#     top_axis.tick_params(
#         axis="x",
#         length=0,
#         pad=8,
#         labelsize=11,
#     )

#     top_axis.spines["top"].set_visible(False)

#     ax.set_xlim(
#         min(positions) - 0.7,
#         max(positions) + 0.7,
#     )

#     fig.tight_layout()

#     output_path = os.path.join(
#         out_dir,
#         "embedding_effect_distance_boxplot_by_K.png",
#     )

#     fig.savefig(
#         output_path,
#         dpi=150,
#         bbox_inches="tight",
#     )

#     plt.close(fig)

#     print(
#         "  Saved embedding_effect_distance_boxplot_by_K.png"
#     )



# plot_metric_gain_boxplots(df_res=df_res, out_dir=out_dir)
# plot_effect_distance_boxplot_by_k(
#         df_res=df_res,
#         out_dir=out_dir,
#     )

In [17]:
def plot_metric_gain_boxplots(
    df_res: pd.DataFrame,
    out_dir: str,
    k_col: str = "K",
) -> None:

    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.patches import Patch

    # ============================================================
    # Methods and display labels
    # ============================================================

    method_order = [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]

    method_labels = {
        "km_greedy": "KM",
        "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
        "joe_km_greedy_linear": "JOECHAR_1LIN",
        "joe_ohe_sigmoid": "JOEOH_3SIG",
        "joe_ohe_linear": "JOEOH_1LIN",
    }

    # ============================================================
    # Fixed colours for methods
    # ============================================================

    palette = sns.color_palette("bright", len(method_order))

    method_colors = {
        method: palette[i]
        for i, method in enumerate(method_order)
    }

    # ============================================================
    # K values
    # ============================================================

    if k_col not in df_res.columns:
        raise KeyError(
            f"Column {k_col!r} not found. "
            f"Available columns: {list(df_res.columns)}"
        )

    k_values = sorted(df_res[k_col].dropna().unique())

    if len(k_values) == 0:
        print("No K values found; skipping gain plot.")
        return

    # ============================================================
    # Metrics
    # ============================================================

    plot_specs = [
        (
            "gain_cindex",
            "C-index gain\n(method - NN OHE)",
        ),
        (
            "gain_ibs",
            "IBS gain\n(NN OHE - method)",
        ),
    ]

    # ============================================================
    # One figure, two rows
    # ============================================================

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(24, 15),
    )

    plotted_methods = set()

    for ax, (prefix, ylabel) in zip(axes, plot_specs):

        data_to_plot = []
        positions = []
        box_colors = []

        group_centres = []
        separator_positions = []

        current_position = 1

        # ========================================================
        # Build groups
        # ========================================================

        for k_index, k_value in enumerate(k_values):

            df_k = df_res.loc[df_res[k_col] == k_value]

            group_positions = []

            for method in method_order:

                column = f"{prefix}_{method}"

                if column not in df_k.columns:
                    continue

                values = (
                    df_k[column]
                    .replace([np.inf, -np.inf], np.nan)
                    .dropna()
                    .to_numpy()
                )

                if len(values) == 0:
                    continue

                data_to_plot.append(values)
                positions.append(current_position)
                box_colors.append(method_colors[method])

                group_positions.append(current_position)
                plotted_methods.add(method)

                current_position += 1

            # Centre of this K group
            if group_positions:

                group_centres.append(
                    np.mean(group_positions)
                )

                # separator AFTER this group
                separator_positions.append(
                    max(group_positions) + 0.5
                )

            # IMPORTANT:
            # no current_position += 1 here
            # -> groups remain close together

        if not data_to_plot:
            continue

        # Last separator is not needed
        separator_positions = separator_positions[:-1]

        # ========================================================
        # Boxplots
        # ========================================================

        boxplot = ax.boxplot(
            data_to_plot,
            positions=positions,
            widths=0.65,
            patch_artist=True,
            showmeans=False,
        )

        # Fill according to method
        for patch, color in zip(
            boxplot["boxes"],
            box_colors,
        ):
            patch.set_facecolor(color)
            patch.set_edgecolor("black")

        for whisker in boxplot["whiskers"]:
            whisker.set_color("black")

        for cap in boxplot["caps"]:
            cap.set_color("black")

        for median in boxplot["medians"]:
            median.set(
                color="black",
                linewidth=2,
            )

        for flier in boxplot["fliers"]:
            flier.set(
                marker="o",
                markerfacecolor="none",
                markeredgecolor="black",
                markersize=4,
            )

        # ========================================================
        # Zero reference
        # ========================================================

        ax.axhline(
            0,
            color="grey",
            linewidth=1,
            linestyle="--",
            zorder=0,
        )

        # ========================================================
        # Vertical separators between K groups
        # ========================================================

        for separator in separator_positions:
            ax.axvline(
                separator,
                color="grey",
                linestyle=":",
                linewidth=1,
            )

        # ========================================================
        # X axis = K only
        # ========================================================


        ax.set_xticks(group_centres)

        ax.set_xticklabels(
            [f"{k} cat" for k in k_values],
            rotation=60,
            ha="right",
            fontsize=23,
        )

        # ========================================================
        # Y axis
        # ========================================================

        ax.set_ylabel(
            ylabel,
            fontsize=23,
        )

        ax.tick_params(
            axis="y",
            labelsize=23,
        )

        ax.grid(
            True,
            axis="y",
            alpha=0.3,
        )

        ax.set_xlim(
            min(positions) - 0.7,
            max(positions) + 0.7,
        )

        # No per-axis legend
        ax.legend([], [], frameon=False)

    # ============================================================
    # Shared method legend
    # ============================================================

    legend_patches = [
        Patch(
            facecolor=method_colors[method],
            edgecolor="black",
            label=method_labels[method],
        )
        for method in method_order
        if method in plotted_methods
    ]

    fig.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=len(legend_patches),
        bbox_to_anchor=(0.5, -0.01),
        fontsize=23,
        frameon=False,
    )

    # Leave room for legend
    plt.tight_layout(
        rect=[0, 0.08, 1, 0.98]
    )

    # ============================================================
    # Save
    # ============================================================

    output_path = os.path.join(
        out_dir,
        "metric_gain_boxplots.pdf",
    )

    fig.savefig(
        output_path,
        bbox_inches="tight",
    )

    plt.close(fig)

    print(f"Saved {output_path}")
def plot_embedding_structure_boxplots(
    df_res: pd.DataFrame,
    out_dir: str,
    k_col: str = "K",
) -> None:
    """
    Plot embedding-structure metrics separately for every K.

    For each metric:
        K = 10                 K = 20                 K = 40
        KM JOECHAR ...         KM JOECHAR ...         KM JOECHAR ...

    Each box contains the seed-level results for that specific
    method and that specific K.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    method_order = [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]

    method_labels = {
        "km_greedy": "KM",
        "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
        "joe_km_greedy_linear": "JOECHAR_1LIN",
        "joe_ohe_sigmoid": "JOEOH_3SIG",
        "joe_ohe_linear": "JOEOH_1LIN",
    }

    plot_specs = [
        (
            "ari",
            "Adjusted Rand index",
            "embedding_ari_boxplot.pdf",
        ),
        (
            "nmi",
            "Normalised mutual information",
            "embedding_nmi_boxplot.pdf",
        ),
        (
            "silhouette_true",
            "True-cluster silhouette score",
            "embedding_silhouette_boxplot.pdf",
        ),
        (
            "effect_distance_spearman",
            "Effect-distance Spearman correlation",
            "embedding_effect_distance_boxplot.pdf",
        ),
    ]

    # ----------------------------------------------------------
    # Check K
    # ----------------------------------------------------------
    if k_col not in df_res.columns:
        raise KeyError(
            f"Column {k_col!r} was not found in df_res. "
            f"Available columns: {list(df_res.columns)}"
        )

    k_values = sorted(
        df_res[k_col].dropna().unique()
    )

    if not k_values:
        print("  No K values found; skipping embedding plots.")
        return

    # ==========================================================
    # One figure per embedding metric
    # ==========================================================
    for metric_prefix, ylabel, filename in plot_specs:

        all_data = []
        positions = []
        tick_labels = []

        group_centres = []
        group_labels = []
        separator_positions = []

        current_position = 1

        # ======================================================
        # LOOP OVER K
        # ======================================================
        for k_value in k_values:

            # Only rows belonging to this K
            df_k = df_res.loc[
                df_res[k_col] == k_value
            ]

            group_positions = []

            # --------------------------------------------------
            # Separate box for every method within this K
            # --------------------------------------------------
            for method in method_order:

                column = f"{metric_prefix}_{method}"

                if column not in df_k.columns:
                    continue

                values = (
                    df_k[column]
                    .replace(
                        [np.inf, -np.inf],
                        np.nan,
                    )
                    .dropna()
                    .to_numpy()
                )

                if len(values) == 0:
                    continue

                all_data.append(values)

                positions.append(
                    current_position
                )

                tick_labels.append(
                    method_labels[method]
                )

                group_positions.append(
                    current_position
                )

                current_position += 1

            # --------------------------------------------------
            # K label and separator
            # --------------------------------------------------
            if group_positions:

                group_centres.append(
                    np.mean(group_positions)
                )

                group_labels.append(
                    f"K = {k_value}"
                )

                separator_positions.append(
                    current_position - 0.5
                )

                # blank space between K groups
                current_position += 1

        # ======================================================
        # Nothing to plot
        # ======================================================
        if not all_data:
            print(
                f"  No valid values found for {metric_prefix}; "
                f"skipping plot."
            )
            continue

        # Remove separator after final K group
        separator_positions = (
            separator_positions[:-1]
        )

        # ======================================================
        # Figure
        # ======================================================
        figure_width = max(
            10,
            0.9 * len(all_data) + 2.5,
        )

        fig, ax = plt.subplots(
            figsize=(figure_width, 5.2)
        )

        boxplot = ax.boxplot(
            all_data,
            positions=positions,
            widths=0.65,
            patch_artist=True,
            showmeans=False,
        )

        # ======================================================
        # Styling
        # ======================================================
        for box in boxplot["boxes"]:
            box.set(
                facecolor="lightgrey",
                edgecolor="black",
                alpha=1.0,
            )

        for whisker in boxplot["whiskers"]:
            whisker.set(
                color="black"
            )

        for cap in boxplot["caps"]:
            cap.set(
                color="black"
            )

        for median in boxplot["medians"]:
            median.set(
                color="black",
                linewidth=2,
            )

        for flier in boxplot["fliers"]:
            flier.set(
                marker="o",
                markerfacecolor="none",
                markeredgecolor="black",
                markersize=4,
            )

        # ======================================================
        # Vertical lines between K groups
        # ======================================================
        for separator in separator_positions:

            ax.axvline(
                separator,
                color="grey",
                linewidth=0.8,
                linestyle="--",
                alpha=0.8,
                zorder=0,
            )

        # ======================================================
        # Method names on bottom
        # ======================================================
        ax.set_xticks(
            positions
        )

        ax.set_xticklabels(
            tick_labels,
            rotation=30,
            ha="right",
        )

        ax.set_ylabel(
            ylabel,
            fontsize=11,
        )

        # ======================================================
        # K labels on top
        # ======================================================
        # top_axis = ax.secondary_xaxis(
        #     "top"
        # )

        # top_axis.set_xticks(
        #     group_centres
        # )

        # top_axis.set_xticklabels(
        #     group_labels
        # )

        # top_axis.tick_params(
        #     axis="x",
        #     length=0,
        #     pad=8,
        #     labelsize=11,
        # )

        # top_axis.spines[
        #     "top"
        # ].set_visible(False)

        ax.set_xlim(
            min(positions) - 0.7,
            max(positions) + 0.7,
        )

        # ------------------------------------------------------
        # Useful reference lines / limits
        # ------------------------------------------------------
        if metric_prefix in ["ari", "nmi"]:
            ax.axhline(
                0,
                color="grey",
                linewidth=0.7,
                linestyle="--",
                zorder=0,
            )

        elif metric_prefix in [
            "silhouette_true",
            "effect_distance_spearman",
        ]:
            ax.axhline(
                0,
                color="grey",
                linewidth=0.7,
                linestyle="--",
                zorder=0,
            )

        fig.tight_layout()

        fig.savefig(
            os.path.join(
                out_dir,
                filename,
            ),
            dpi=150,
            bbox_inches="tight",
        )

        plt.close(fig)

        print(
            f"  Saved {filename}"
        )

plot_metric_gain_boxplots(df_res=df_res, out_dir=out_dir)
plot_embedding_structure_boxplots(
        df_res=df_res,
        out_dir=out_dir,
    )

Saved results_ch5_comparison_cluster/metric_gain_boxplots.pdf
  Saved embedding_ari_boxplot.pdf
  Saved embedding_nmi_boxplot.pdf
  Saved embedding_silhouette_boxplot.pdf
  Saved embedding_effect_distance_boxplot.pdf


In [18]:
import shutil
import os

source_dir = out_dir  # out_dir is defined in the previous cell as 'results_ch5_comparison_cluster'
if EXPERIMENT_TYPE == "cluster":
    destination_dir = "/content/drive/MyDrive/Colab Notebooks/results_ch5_comparison_cluster"
else:
    destination_dir = "/content/drive/MyDrive/Colab Notebooks/results_ch5_comparison_noncluster"

# Remove destination if it already exists to allow fresh copy
if os.path.exists(destination_dir):
    shutil.rmtree(destination_dir)
    print(f"Removed existing directory: {destination_dir}")

# Copy the directory to Google Drive
shutil.copytree(source_dir, destination_dir)
print(f"All outputs from '{source_dir}' saved to '{destination_dir}'")

Removed existing directory: /content/drive/MyDrive/Colab Notebooks/results_ch5_comparison_cluster
All outputs from 'results_ch5_comparison_cluster' saved to '/content/drive/MyDrive/Colab Notebooks/results_ch5_comparison_cluster'
